# ML Study Tracker 3: Deep Learning, GenAI, and MLOps

Split from `ml_study_tracker-clauded.ipynb`.


# Table of Contents

- [Deep Learning and GenAI](#deep-learning-and-genai)
  - [Neural Network Basics](#neural-network-basics)
  - [CNNs](#cnns)
  - [RNNs / LSTMs](#rnns--lstms)
  - [Transformers](#transformers)
  - [NLP Basics](#nlp-basics)
  - [Embeddings](#embeddings)
  - [LLM Basics](#llm-basics)
  - [RAG](#rag)
  - [AI Agents](#ai-agents)
  - [Generative Models](#generative-models)
  - [Reinforcement Learning](#reinforcement-learning)
- [Deployment and MLOps](#deployment-and-mlops)
  - [Model Deployment Basics](#model-deployment-basics)
  - [MLOps Basics](#mlops-basics)
- [Mini Project Tracker](#mini-project-tracker)
- [Experiment Log](#experiment-log)


In [1]:
# Common imports

try:
    import numpy as np
except ImportError:
    np = None

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        confusion_matrix, classification_report,
        mean_absolute_error, mean_squared_error, r2_score
    )
except ImportError:
    train_test_split = cross_val_score = GridSearchCV = None
    accuracy_score = precision_score = recall_score = f1_score = None
    confusion_matrix = classification_report = None
    mean_absolute_error = mean_squared_error = r2_score = None

# Add more imports as needed.


# Deep Learning and GenAI

## Neural Network Basics

### Study checklist
- [ ] Neuron intuition
- [ ] Activation functions
- [ ] Forward pass
- [ ] Backpropagation intuition
- [ ] Loss functions
- [ ] Optimizers
- [ ] Overfitting and regularization

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Neuron intuition

**Approach:**
- Protects representation and model fit: a neuron's weighted sum plus non-linear activation is the basic unit that lets stacked layers approximate arbitrarily complex functions instead of just linear ones.
- Why this is the right approach: a neuron computes sum(x_i*w_i)+b, mathematically identical to the linear/logistic regression equation covered elsewhere - passing that sum through a non-linear activation is what breaks the equivalence to a purely linear model, since composing multiple purely linear layers would collapse back into just one linear function no matter how many layers were stacked.
- For this checklist item: A neuron computes a weighted sum of inputs then applies a non-linear activation function; stacking layers lets the network learn hierarchical representations.
- Code walkthrough: Check that pre_activation is the raw weighted sum plus bias, and that post_activation clips it to 0 if negative - ReLU zeroing out a negative pre-activation.

**Learn more:**
- Website: [Dive into Deep Learning: Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Neural+Network+Basics+Neuron+intuition+machine+learning+theory)

**Trade-offs:**
- Compared to a decision tree's if/else splits, a neuron's weighted sum plus activation is a smooth, differentiable unit - exactly what makes gradient-based learning possible, but also why a single neuron alone can't express a hard rule the way a tree split can.
- A single neuron is just logistic regression in disguise; the expressive power only shows up once you stack many of them, which is also where the tuning and compute cost start.

**Practical software engineering use cases:**
- When to use it: Use this to build intuition before touching a deep learning framework - trace a single neuron's forward computation by hand on a toy input.
- When not to use it: Don't use a single neuron, or a shallow network of a few, for a problem simple tabular models like logistic regression already solve - the extra machinery buys nothing at that scale.


In [ ]:
# Neural Network Basics - Neuron intuition
def relu(x):
    return max(0, x)

inputs = [0.5, -1.0, 2.0]
weights = [0.2, 0.8, -0.5]
bias = 0.1
pre_activation = sum(x * w for x, w in zip(inputs, weights)) + bias
post_activation = relu(pre_activation)
print("weighted sum + bias:", round(pre_activation, 3))
print("after ReLU:", round(post_activation, 3))


### Checklist item: Activation functions

**Approach:**
- Protects model fit and trainability: activations introduce the non-linearity that lets a network learn more than a linear mapping, and their shape directly affects how gradients flow during training.
- Why this is the right approach: if every layer's activation were linear, a composition of them would collapse mathematically into a single linear function no matter how many layers exist - the specific non-linearity of ReLU, sigmoid, or tanh is what prevents that collapse, precisely why non-linear activations are required for a multi-layer network to represent anything a single linear layer couldn't already represent alone.
- For this checklist item: ReLU (max(0,x)) is fast and avoids vanishing gradients; sigmoid squashes to (0,1) for output probabilities; tanh squashes to (-1,1) for hidden layers.
- Code walkthrough: Check how sigmoid squashes every x into (0,1), tanh into (-1,1), and relu clips negatives to exactly 0 - three different output shapes from the same three input values.

**Learn more:**
- Website: [Dive into Deep Learning: Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Neural+Network+Basics+Activation+functions+machine+learning+theory)

**Trade-offs:**
- Compared to a linear model's fixed shape, choosing an activation function is an extra design decision every hidden layer requires - one linear regression and logistic regression never have to make.
- ReLU is cheap and avoids vanishing gradients for positive inputs but can permanently output zero for large negative inputs; sigmoid and tanh are smooth but saturate and vanish gradients in deep networks, so pick based on depth and layer position, not habit.

**Practical software engineering use cases:**
- When to use it: Use ReLU as the default for hidden layers in a new architecture, and reserve sigmoid/tanh for output layers or gates where a bounded range is specifically needed.
- When not to use it: Don't use sigmoid or tanh throughout a very deep network by default - their gradients vanish across many layers in a way ReLU-family activations largely avoid.


In [ ]:
# Neural Network Basics - Activation functions
import math

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def relu(x):
    return max(0, x)

for x in [-2, 0, 2]:
    print({"x": x, "sigmoid": round(sigmoid(x), 3), "tanh": round(math.tanh(x), 3), "relu": relu(x)})


### Checklist item: Forward pass

**Approach:**
- Protects correctness and debuggability: the forward pass is the deterministic computation from input to output, so being able to trace it by hand on a toy example is what lets you verify a model is wired correctly before trusting its training.
- Why this is the right approach: a forward pass is repeated application of the same weighted-sum-plus-activation formula from one layer, using the previous layer's output as the next layer's input - a mathematically well-defined composition of functions, which is why tracing it by hand on a tiny network works: each step is just the same simple formula applied again.
- For this checklist item: The forward pass propagates inputs through each layer in order â€” weights, biases, activations â€” to produce a prediction; no learning happens here, only inference.
- Code walkthrough: Check that hidden is computed by applying weights, bias, and ReLU to the input features, then that same hidden vector feeds into the second layer to produce logit - two chained layers, hand-traced.

**Learn more:**
- Website: [Dive into Deep Learning: Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Neural+Network+Basics+Forward+pass+machine+learning+theory)

**Trade-offs:**
- Tracing a forward pass by hand is the deep learning equivalent of stepping through a decision tree's splits - both build the same kind of predictable-computation intuition, just for very different kinds of models.
- Computing it manually is only feasible on tiny toy networks; at real scale you rely on the framework's autograd and shape-checking instead, so invest in unit tests on toy inputs rather than manual verification of the full model.

**Practical software engineering use cases:**
- When to use it: Use a hand-traced forward pass on a 2-3 neuron toy network when debugging a shape mismatch or unexpected output in a real model.
- When not to use it: Don't hand-verify a real, multi-layer model's forward pass this way - rely on the framework's shape-checking and a unit test on a tiny synthetic batch instead.


In [ ]:
# Neural Network Basics - Forward pass
def relu(x):
    return max(0, x)

features = [1.0, -2.0]
hidden_weights = [[0.5, -0.3], [0.8, 0.2]]
hidden_bias = [0.1, -0.2]
hidden = [relu(sum(x * w for x, w in zip(features, weights)) + b) for weights, b in zip(hidden_weights, hidden_bias)]
output_weights = [1.2, -0.7]
logit = sum(h * w for h, w in zip(hidden, output_weights)) + 0.05
print("hidden activations:", [round(v, 3) for v in hidden])
print("output logit:", round(logit, 3))


### Checklist item: Backpropagation intuition

**Approach:**
- Protects trainability: backpropagation is the chain-rule computation that turns a single scalar loss into gradients for every parameter, which is what makes training deep networks computationally feasible at all.
- Why this is the right approach: the chain rule states the derivative of a composed function f(g(x)) is f'(g(x))*g'(x) - backpropagation is exactly this rule applied repeatedly, layer by layer, working backward from the loss, which is why the gradient for any weight can always be computed as a product of local derivatives along the path from that weight to the loss, no matter how deep the network is.
- For this checklist item: Backpropagation applies the chain rule from the loss back through each operation, producing gradients for weights and biases so an optimizer can update them.
- Code walkthrough: Check that grad_w equals dloss_dpred times dpred_dw exactly as the chain-rule comment states - that product is literally what backprop computes for this one-weight network.

**Learn more:**
- Website: [Dive into Deep Learning: Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Neural+Network+Basics+Backpropagation+intuition+machine+learning+theory)

**Trade-offs:**
- Unlike a tree ensemble's feature importances, which are read off directly after training, understanding backpropagation requires following the chain rule through every layer - a fundamentally different, and initially less intuitive, kind of transparency.
- Understanding it conceptually matters for debugging, for example recognizing vanishing or exploding gradients, but you should never hand-implement it for real models; use the framework's autograd and reserve manual backprop for small pedagogical examples.

**Practical software engineering use cases:**
- When to use it: Use this understanding specifically when a training run's loss doesn't move or explodes - that's when knowing what backprop actually computes tells you where to look.
- When not to use it: Don't reach for manual backprop math to debug a real production model - check the framework's gradient magnitudes and norms directly instead.


In [ ]:
# Neural Network Basics - Backpropagation intuition
x = 2.0
y_true = 5.0
w = 1.5
prediction = w * x
loss = (prediction - y_true) ** 2
# Chain rule: dloss/dw = dloss/dpred * dpred/dw
dloss_dpred = 2 * (prediction - y_true)
dpred_dw = x
grad_w = dloss_dpred * dpred_dw
print("prediction:", prediction)
print("loss:", loss)
print("gradient for w:", grad_w)


### Checklist item: Loss functions

**Approach:**
- Protects evaluation integrity and model fit: the loss function is literally what the optimizer minimizes, so choosing the wrong one, such as MSE for a classification problem, optimizes the wrong thing even if the architecture is fine.
- Why this is the right approach: cross-entropy is derived directly from the mathematical definition of the log-likelihood of the correct class under the model's predicted probability distribution - maximizing that likelihood, equivalently minimizing its negative log, is the formally optimal way to fit a probability model to observed data, which is why cross-entropy isn't an arbitrary choice for classification.
- For this checklist item: Choose a loss that matches the prediction task: regression commonly uses squared/absolute error, while classification commonly uses cross-entropy or log loss.
- Code walkthrough: Check that mse only depends on the numeric gap between prediction and target, while binary_cross_entropy depends on log-probabilities - two different loss shapes for two different task types.

**Learn more:**
- Website: [Dive into Deep Learning: Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Neural+Network+Basics+Loss+functions+machine+learning+theory)

**Trade-offs:**
- Unlike scikit-learn's estimators, which mostly hide the loss function behind a single fit() call, deep learning frameworks make you choose and often write the loss explicitly - more upfront decisions, but real flexibility when you need a custom objective.
- The loss you optimize during training and the metric you report to stakeholders, such as accuracy or F1, don't have to be the same and often shouldn't be, since losses need to be differentiable while metrics need to be interpretable.

**Practical software engineering use cases:**
- When to use it: Use cross-entropy for classification and MSE, or a robust variant like Huber, for regression as your starting points, then customize only if the standard choice doesn't match your real objective.
- When not to use it: Don't pick a loss function purely because it looks standard for the task type without checking it against your actual evaluation metric - a mismatch there is quiet but costly.


In [ ]:
# Neural Network Basics - Loss functions
import math

y_reg_true, y_reg_pred = 10.0, 8.5
mse = (y_reg_pred - y_reg_true) ** 2

y_class_true, prob_positive = 1, 0.8
binary_cross_entropy = -(y_class_true * math.log(prob_positive) + (1 - y_class_true) * math.log(1 - prob_positive))
print("regression MSE:", round(mse, 3))
print("binary cross-entropy:", round(binary_cross_entropy, 3))


### Checklist item: Optimizers

**Approach:**
- Protects trainability and training efficiency: the optimizer determines how parameters move in response to gradients, directly affecting convergence speed and whether training gets stuck in poor local minima or plateaus.
- Why this is the right approach: SGD's plain update only uses the CURRENT gradient; momentum's update accumulates a running weighted average of past gradients first, mathematically smoothing out oscillations in directions where the gradient repeatedly flips sign while reinforcing consistent directions - a precise, derivable mechanism for faster convergence, not just an empirical trick.
- For this checklist item: Optimizers turn gradients into parameter updates; SGD uses the raw gradient, while momentum/Adam-style methods smooth or adapt updates for faster, steadier training.
- Code walkthrough: Check that w_momentum moves differently than w_sgd across the same three gradients - momentum accumulates velocity from previous gradients instead of reacting only to the current one.

**Learn more:**
- Website: [Dive into Deep Learning: Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Neural+Network+Basics+Optimizers+machine+learning+theory)

**Trade-offs:**
- Unlike classical ML models trained by a closed-form solution or a single scikit-learn solver choice, a neural network's optimizer is itself a meaningful design decision with real accuracy and speed consequences.
- Adam converges fast and needs less learning-rate tuning, making it a safe default, but plain SGD with momentum can generalize better on some tasks, such as image classification, at the cost of needing a carefully tuned learning-rate schedule.

**Practical software engineering use cases:**
- When to use it: Use Adam as your default optimizer when starting a new model, especially early in development when you want fast, low-maintenance convergence.
- When not to use it: Don't assume Adam is strictly better once you're optimizing for final accuracy on an established architecture - well-tuned SGD with momentum sometimes generalizes better.


In [ ]:
# Neural Network Basics - Optimizers
gradients = [-4.0, -2.0, -1.0]
learning_rate = 0.1
w_sgd = 0.0
w_momentum = 0.0
velocity = 0.0
for grad in gradients:
    w_sgd -= learning_rate * grad
    velocity = 0.9 * velocity + grad
    w_momentum -= learning_rate * velocity
    print({"grad": grad, "sgd_w": round(w_sgd, 3), "momentum_w": round(w_momentum, 3)})


### Checklist item: Overfitting and regularization

**Approach:**
- Protects evaluation integrity and generalization: deep networks have enough capacity to memorize training data outright, so regularization such as dropout, weight decay, and early stopping is what keeps the model learning generalizable patterns instead.
- Why this is the right approach: L2 weight decay adds alpha*sum(w^2) to the loss being minimized, and its derivative with respect to each weight is proportional to that weight's own current value - this mathematically pulls every weight toward zero at each update step, shrinking the model's effective capacity, precisely the mechanism by which it counteracts a network's tendency to fit noise given enough free parameters.
- For this checklist item: Dropout randomly zeros activations during training so no single neuron is relied upon; L2 weight decay (weight_decay) penalizes large weights â€” both reduce overfitting.
- Code walkthrough: Check that gap turns positive and grows in the later epochs, with validation loss rising while train loss keeps falling - that widening gap is the numeric definition of overfitting.

**Learn more:**
- Website: [Dive into Deep Learning: Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Neural+Network+Basics+Overfitting+and+regularization+machine+learning+theory)

**Trade-offs:**
- Neural networks need explicit regularization like dropout and weight decay far more than a shallow model like logistic regression, precisely because their much larger parameter count gives them more room to memorize.
- Each regularization technique trades some training-set fit for validation-set generalization; stacking too many at once, such as heavy dropout plus heavy weight decay plus aggressive augmentation, can underfit, so tune them incrementally against a validation set.

**Practical software engineering use cases:**
- When to use it: Use dropout and/or weight decay by default on any network with enough capacity to plausibly memorize your training set.
- When not to use it: Don't add every regularization technique at once to be safe on a small or already-underfitting model - stacked regularization can push it straight into underfitting.


In [ ]:
# Neural Network Basics - Overfitting and regularization
train_loss = [0.8, 0.45, 0.25, 0.12]
validation_loss = [0.82, 0.50, 0.42, 0.55]
weights = [1.5, -2.0, 0.5]
l2_penalty = 0.01 * sum(w * w for w in weights)
for epoch, (tr, val) in enumerate(zip(train_loss, validation_loss), start=1):
    gap = val - tr
    print(f"epoch={epoch} train={tr:.2f} val={val:.2f} gap={gap:.2f}")
print("L2 penalty added to loss:", round(l2_penalty, 3))


In [248]:
# Practice: Neural Network Basics

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## CNNs

### Study checklist
- [ ] Convolution
- [ ] Filters/kernels
- [ ] Pooling
- [ ] Image classification
- [ ] Transfer learning
- [ ] When CNNs are useful

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Convolution

**Approach:**
- Protects representation: convolution shares weights across spatial locations, which encodes the assumption that a pattern, like an edge, means the same thing wherever it appears in the image.
- Why this is the right approach: convolution is mathematically defined as sliding a fixed small kernel across the input and computing a weighted sum at each position - reusing the SAME kernel weights at every position is what encodes translation invariance directly into the operation's definition, meaning a pattern detected in one location is mathematically detected the same way anywhere else in the image.
- For this checklist item: Convolution slides a small kernel across the input and computes dot products at each position â€” this detects local patterns regardless of where they appear.
- Code walkthrough: Check that each entry in features is the kernel's weighted sum over a sliding 3-element window of signal - that sliding-window slicing is exactly what convolution does.

**Learn more:**
- Website: [Dive into Deep Learning: Convolutional Neural Networks](https://d2l.ai/chapter_convolutional-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=CNNs+Convolution+machine+learning+theory)

**Trade-offs:**
- Unlike a fully-connected layer, where every input pixel gets its own independent weight, convolution reuses the same small set of weights across the whole image - a structural assumption regular neural network layers don't make.
- Weight sharing is what makes CNNs parameter-efficient and translation-invariant, but that same assumption fails for data without spatial locality, such as unordered tabular features, so don't reach for convolution there.

**Practical software engineering use cases:**
- When to use it: Use convolutional layers as your default first layers for any image, spectrogram, or other grid-structured input.
- When not to use it: Don't apply convolution to features with no spatial or sequential order, like a table of unrelated customer attributes - there's no locality for it to exploit.


In [249]:
# CNNs - Convolution
signal = [1, 2, 3, 4]
kernel = [0.25, 0.5, 0.25]
features = []
for i in range(len(signal) - len(kernel) + 1):
    window = signal[i:i + len(kernel)]
    features.append(sum(x * w for x, w in zip(window, kernel)))
print("convolved features:", features)


convolved features: [2.0, 3.0]


### Checklist item: Filters/kernels

**Approach:**
- Protects representation: each filter learns to detect one specific local pattern, such as an edge or a texture, and stacking layers of filters lets the network build from simple patterns to complex ones hierarchically.
- Why this is the right approach: each filter's output at a position is the dot product between the filter's fixed weights and the local patch of input values there - because a dot product is maximized when two vectors point in the same direction, a filter's weights mathematically define exactly the local pattern that will produce its highest activation, which is why different weight patterns detect different features.
- For this checklist item: Each filter (kernel) learns to detect one pattern (edge, texture, shape); deeper layers combine lower-level filters to detect higher-level concepts.
- Code walkthrough: Check that 'edge_detector' produces large values exactly where image_row jumps from 10 to 30, while 'smoother' produces intermediate averaged values - two different kernels extracting two different patterns from the same input.

**Learn more:**
- Website: [Dive into Deep Learning: Convolutional Neural Networks](https://d2l.ai/chapter_convolutional-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=CNNs+Filters%2Fkernels+machine+learning+theory)

**Trade-offs:**
- Unlike a decision tree, where what a rule detects is a single feature threshold, a filter's learned pattern only becomes interpretable by visualizing what activates it - a much less direct form of transparency.
- More filters per layer increase the model's capacity to represent diverse patterns, but also increase compute and overfitting risk; filter count is a capacity knob to tune like any other, not something to maximize by default.

**Practical software engineering use cases:**
- When to use it: Use more filters in deeper layers of a CNN, where the network needs to represent increasingly complex, composite patterns.
- When not to use it: Don't increase filter counts as a first fix for poor accuracy - check data quality and architecture depth before assuming you need more capacity per layer.


In [ ]:
# CNNs - Filters/kernels
image_row = [10, 10, 30, 30]
filters = {
    "edge_detector": [-1, 1],
    "smoother": [0.5, 0.5],
}
for name, kernel in filters.items():
    features = [sum(x * w for x, w in zip(image_row[i:i + len(kernel)], kernel)) for i in range(len(image_row) - len(kernel) + 1)]
    print(name, features)


### Checklist item: Pooling

**Approach:**
- Protects model fit and computational efficiency: pooling downsamples feature maps, which reduces computation and adds a degree of translation invariance so the model tolerates small shifts in where a pattern appears.
- Why this is the right approach: max pooling is defined as taking the maximum value within a fixed window, a specific, parameter-free mathematical reduction that discards exact position within the window while retaining whether the feature was present at all in that neighborhood - precisely the defined trade-off between spatial precision and translation tolerance the operation makes by construction.
- For this checklist item: Max-pooling keeps the strongest activation in each region, reducing spatial size and adding translation invariance â€” typical after each conv block.
- Code walkthrough: Check that max_pooled keeps only the larger value from each pair while avg_pooled averages the pair - same windows, two different summary rules.

**Learn more:**
- Website: [Dive into Deep Learning: Convolutional Neural Networks](https://d2l.ai/chapter_convolutional-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=CNNs+Pooling+machine+learning+theory)

**Trade-offs:**
- Compared to a strided convolution, which also downsamples but keeps learnable weights in the process, pooling is a fixed, parameter-free operation - cheaper, but with no ability to learn what to preserve.
- Pooling discards spatial precision along with the noise it's meant to suppress, so tasks needing precise localization, such as segmentation or detection, often reduce or replace pooling with strided convolutions instead.

**Practical software engineering use cases:**
- When to use it: Use pooling in a classification CNN where only whether a pattern is present somewhere in the image matters, not exactly where.
- When not to use it: Don't use aggressive pooling in a segmentation or detection model that needs to localize objects precisely - it discards the spatial precision those tasks depend on.


In [ ]:
# CNNs - Pooling
feature_map = [1.0, 3.5, 2.0, 4.0]
window = 2
max_pooled = [max(feature_map[i:i + window]) for i in range(0, len(feature_map), window)]
avg_pooled = [sum(feature_map[i:i + window]) / window for i in range(0, len(feature_map), window)]
print("max pooled:", max_pooled)
print("avg pooled:", avg_pooled)


### Checklist item: Image classification

**Approach:**
- Protects model fit and evaluation integrity: this is the task CNNs were originally designed around, so it's the benchmark for confirming an architecture, augmentation strategy, and training loop are all working correctly.
- Why this is the right approach: stacking convolution and pooling layers repeatedly composes many simple local operations into a function capable of representing increasingly complex, spatially hierarchical patterns - a mathematical consequence of function composition, which is why deeper CNNs can represent more complex visual concepts than a single convolutional layer.
- For this checklist item: Global average pooling or flattening after conv layers feeds into a dense classifier; the conv layers act as a learned feature extractor.
- Code walkthrough: Check that pooled reduces each feature map to a single max value, and that scores combines those pooled features with each class's learned weights to pick a prediction.

**Learn more:**
- Website: [Dive into Deep Learning: Convolutional Neural Networks](https://d2l.ai/chapter_convolutional-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=CNNs+Image+classification+machine+learning+theory)

**Trade-offs:**
- Unlike tabular classification, where feature engineering often does most of the work, image classification leans almost entirely on the architecture and training data to discover useful features on its own.
- A CNN trained from scratch needs substantial labeled data to reach good accuracy; with limited data, transfer learning from a pretrained model almost always outperforms training from scratch.

**Practical software engineering use cases:**
- When to use it: Use a CNN, or transfer learning on top of one, as the default approach for any image classification task with reasonable amounts of labeled data.
- When not to use it: Don't train a CNN from scratch on a small labeled image dataset (a few hundred examples) - transfer learning from a pretrained model will almost always do better with the same data.


In [ ]:
# CNNs - Image classification
feature_maps = {
    "edge_strength": [0.2, 0.8, 0.6],
    "texture_strength": [0.1, 0.4, 0.9],
}
pooled = {name: max(values) for name, values in feature_maps.items()}
class_weights = {"cat": {"edge_strength": 0.4, "texture_strength": 0.9}, "road": {"edge_strength": 0.8, "texture_strength": 0.2}}
scores = {label: sum(pooled[name] * weight for name, weight in weights.items()) for label, weights in class_weights.items()}
print("pooled features:", pooled)
print("class scores:", scores)
print("prediction:", max(scores, key=scores.get))


### Checklist item: Transfer learning

**Approach:**
- Protects data efficiency and model fit: reusing features learned on a large dataset, like ImageNet, lets you reach good accuracy on a small, domain-specific dataset that couldn't train a deep network from scratch on its own.
- Why this is the right approach: freezing a pretrained layer means fixing its weights so its gradient is never applied during training, mathematically excluding it from the optimization entirely - only the unfrozen layers' weights are updated by the loss's gradient, precisely why transfer learning can reuse a pretrained layer's learned function exactly, while still adapting the rest of the network to a new task.
- For this checklist item: Transfer learning starts from ImageNet-pretrained weights; freezing early layers and fine-tuning later ones adapts the network to a new task with little data.
- Code walkthrough: Check that frozen_features are computed once using the fixed pretrained_filters, and that only task_head_weights, not the filters, participate in computing the final task-specific score.

**Learn more:**
- Website: [Dive into Deep Learning: Convolutional Neural Networks](https://d2l.ai/chapter_convolutional-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=CNNs+Transfer+learning+machine+learning+theory)

**Trade-offs:**
- Unlike training a model from scratch, which needs a large labeled dataset, transfer learning lets you reuse features learned on someone else's massive dataset - shifting the bottleneck from data volume to compatibility between the pretrained domain and yours.
- Freezing early layers and fine-tuning only the head is fast and low-risk of overfitting on small data, but fine-tuning more layers can improve accuracy further if you have enough data to avoid catastrophically forgetting the pretrained features.

**Practical software engineering use cases:**
- When to use it: Use transfer learning as your default starting point for any new image classification task, freezing early layers first and unfreezing more only if accuracy plateaus.
- When not to use it: Don't fine-tune every layer of a large pretrained model on a very small dataset - you risk catastrophically overwriting the useful pretrained features with noise from too few examples.


In [253]:
# CNNs - Transfer learning
image_row = [10, 12, 18, 30, 31]
pretrained_filters = {
    "edge_detector": [-1, 0, 1],
    "smoother": [0.25, 0.5, 0.25],
}

def convolve_1d(values, kernel):
    return [
        sum(x * w for x, w in zip(values[i:i + len(kernel)], kernel))
        for i in range(len(values) - len(kernel) + 1)
    ]

# Transfer learning: freeze reusable filters, train only a small task head.
frozen_features = {
    name: convolve_1d(image_row, kernel)
    for name, kernel in pretrained_filters.items()
}
task_head_weights = {"edge_detector": 0.7, "smoother": 0.3}
score = sum(task_head_weights[name] * values[-1] for name, values in frozen_features.items())

print("frozen features:", frozen_features)
print("new task score:", round(score, 2))


frozen features: {'edge_detector': [8, 18, 13], 'smoother': [13.0, 19.5, 27.25]}
new task score: 17.27


### Checklist item: When CNNs are useful

**Approach:**
- Protects deployment reliability: knowing CNNs' sweet spot, namely data with spatial or local structure like images, spectrograms, or grid-like data, prevents applying convolution where there's no spatial structure to exploit.
- Why this is the right approach: convolution's weight-sharing assumption is only a mathematically valid inductive bias when a local pattern's meaning genuinely doesn't depend on its position - for tabular data where each column has a distinct, fixed meaning, that translation-invariance assumption doesn't correspond to any real property of the data, so there's no mathematical benefit to local weight-sharing there.
- For this checklist item: CNNs work best on grid-structured data with local spatial or temporal correlations (images, spectrograms, EEG); they are not the right choice for tabular data.
- Code walkthrough: Check that only datasets with both grid=True and local_patterns=True get useful=True - that's the specific structural requirement CNNs depend on.

**Learn more:**
- Website: [Dive into Deep Learning: Convolutional Neural Networks](https://d2l.ai/chapter_convolutional-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=CNNs+When+CNNs+are+useful+machine+learning+theory)

**Trade-offs:**
- Compared to a Transformer processing the same image as patches, a CNN's built-in locality assumption needs less data to start learning useful features - an advantage that shrinks as pretraining data grows to internet scale.
- CNNs excel on grid-structured data but need meaningful spatial locality to pay off; on tabular data or short unordered feature vectors, a simpler model usually matches or beats a CNN with far less tuning.

**Practical software engineering use cases:**
- When to use it: Use a CNN when you have a moderate amount of image data and want strong performance without needing internet-scale pretraining.
- When not to use it: Don't use a CNN for text, tabular, or otherwise non-grid data expecting the same spatial-locality benefit - reach for a model whose structure matches the data's actual structure instead.


In [ ]:
# CNNs - When CNNs are useful
datasets = {
    "image pixels": {"grid": True, "local_patterns": True},
    "audio spectrogram": {"grid": True, "local_patterns": True},
    "customer table": {"grid": False, "local_patterns": False},
}
for name, traits in datasets.items():
    useful = traits["grid"] and traits["local_patterns"]
    print(name, "CNN useful=" + str(useful))


In [255]:
# Practice: CNNs

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## RNNs / LSTMs

### Study checklist
- [ ] Sequential data
- [ ] Hidden state
- [ ] Vanishing gradients
- [ ] LSTM gates intuition
- [ ] Text and time-series use cases
- [ ] Limitations compared to transformers

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Sequential data

**Approach:**
- Protects representation: RNNs process inputs one step at a time while carrying forward a hidden state, which is what lets them model order and dependency between elements that a feedforward network would treat as independent.
- Why this is the right approach: an RNN's hidden state at step t is mathematically defined as a function of BOTH the current input and the previous hidden state - this recursive definition is what makes the network's output at any step depend on the entire history of inputs that came before it, a property no feedforward function of a single input alone can have.
- For this checklist item: RNNs process sequences by maintaining a hidden state that summarizes the history seen so far â€” each step reads the current input and the previous hidden state.
- Code walkthrough: Check that prefix grows by one token at each step - each step's output can only see the tokens that came before it, never ones that come after.

**Learn more:**
- Website: [Dive into Deep Learning: Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RNNs+%2F+LSTMs+Sequential+data+machine+learning+theory)

**Trade-offs:**
- Unlike a CNN, which processes an entire input in parallel, or a feedforward network, which assumes independent inputs, an RNN's step-by-step processing is specifically what lets it model order - at the direct cost of that same step-by-step speed.
- Processing sequentially prevents parallelization across time steps during training, which is a major reason transformers, which process a sequence in parallel via attention, have largely replaced RNNs for long sequences.

**Practical software engineering use cases:**
- When to use it: Use an RNN/LSTM when you need to model strict sequential dependency and don't have the compute budget or data scale that would justify a transformer.
- When not to use it: Don't use a plain feedforward or CNN architecture on data where order matters, like a sentence or a time series - you'd be discarding the exact signal that makes the sequence meaningful.


In [ ]:
# RNNs / LSTMs - Sequential data
sequence = ["I", "love", "ML"]
for t, token in enumerate(sequence, start=1):
    prefix = sequence[:t]
    print(f"step={t} current={token} prefix={prefix}")


### Checklist item: Hidden state

**Approach:**
- Protects representation: the hidden state is the RNN's compressed summary of everything seen so far in the sequence, so its capacity, or size, directly limits how much history the model can actually retain.
- Why this is the right approach: because h_t is computed from h_(t-1) using a FIXED-SIZE vector regardless of how many steps have already occurred, all information about the entire past must be compressed into that same fixed number of dimensions - a mathematical bottleneck built into the recurrence's definition, not a limitation of any particular hidden-state size chosen.
- For this checklist item: The hidden state is the RNN's memory; it is passed to the next time step and (optionally) to the output layer â€” it compresses the entire past sequence into a fixed vector.
- Code walkthrough: Check that hidden at each step depends on both the current value and the previous hidden state via the 0.5*hidden term - that recurrence is what carries information forward through the sequence.

**Learn more:**
- Website: [Dive into Deep Learning: Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RNNs+%2F+LSTMs+Hidden+state+machine+learning+theory)

**Trade-offs:**
- Compared to attention, which can look back at any earlier position directly, an RNN's hidden state must compress the entire history into one fixed-size vector - a much tighter memory bottleneck.
- A larger hidden state can retain more context but is more expensive to compute and more prone to overfitting on limited sequence data; it's also still a fixed-size bottleneck, unlike attention which can look back at any position directly.

**Practical software engineering use cases:**
- When to use it: Use a larger hidden state when your sequences carry meaningfully more information to retain, such as longer documents or richer per-step signals.
- When not to use it: Don't assume a bigger hidden state alone will fix an RNN struggling with long sequences - the fixed-size bottleneck and vanishing gradients are structural limits, not just a capacity one.


In [ ]:
# RNNs / LSTMs - Hidden state
import math
sequence = [1.0, 0.5, -0.5]
hidden = 0.0
for t, value in enumerate(sequence, start=1):
    hidden = math.tanh(value + 0.5 * hidden)
    print(f"step={t} hidden_state={hidden:.3f}")


### Checklist item: Vanishing gradients

**Approach:**
- Protects trainability: in vanilla RNNs, gradients shrink exponentially as they're backpropagated through many time steps, which is what makes it hard for the network to learn dependencies between distant elements in a sequence.
- Why this is the right approach: backpropagation through T time steps involves multiplying T derivative terms together via the chain rule applied repeatedly through the recurrence - if each term has magnitude less than 1, their product shrinks exponentially with T, a direct mathematical consequence of repeated multiplication of small numbers, precisely why gradients from distant time steps vanish in a plain RNN.
- For this checklist item: Repeated multiplication through many time steps can shrink gradients toward zero, making early sequence positions hard to learn from.
- Code walkthrough: Check how fast contribution shrinks toward 0 as steps_back grows from 1 to 20 - that's a recurrent gradient of 0.5 compounding multiplicatively, the literal mechanism behind vanishing gradients.

**Learn more:**
- Website: [Dive into Deep Learning: Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RNNs+%2F+LSTMs+Vanishing+gradients+machine+learning+theory)

**Trade-offs:**
- Unlike a well-behaved feedforward network of moderate depth, a vanilla RNN's gradients shrink multiplicatively at every one of potentially hundreds of time steps - a much more severe version of the same underlying problem.
- Architectures like LSTM and GRU mitigate this with gating mechanisms, and gradient clipping helps with the reverse problem of exploding gradients, but neither fully eliminates the difficulty of learning very long-range dependencies, which is a core reason attention-based models were developed.

**Practical software engineering use cases:**
- When to use it: Use LSTM or GRU cells, not vanilla RNN cells, any time your sequences are long enough that distant context plausibly matters.
- When not to use it: Don't use a vanilla RNN on long sequences expecting it to learn long-range dependencies - its gradients will have vanished long before training signal reaches distant time steps.


In [ ]:
# RNNs / LSTMs - Vanishing gradients
recurrent_gradient = 0.5
for steps_back in [1, 2, 5, 10, 20]:
    contribution = recurrent_gradient ** steps_back
    print(f"{steps_back:>2} steps back -> gradient scale {contribution:.6f}")


### Checklist item: LSTM gates intuition

**Approach:**
- Protects representation and trainability: the forget, input, and output gates let an LSTM learn what to keep, update, or discard in its memory cell, which is what allows it to retain information over much longer sequences than a vanilla RNN.
- Why this is the right approach: the LSTM's cell state update is an ADDITIVE combination rather than the repeated multiplicative composition a plain RNN uses - addition doesn't shrink exponentially the way repeated multiplication by numbers less than 1 does, the specific mathematical reason LSTMs can preserve gradient signal over far more time steps than a vanilla RNN.
- For this checklist item: LSTM gates (forget, input, output) control what information flows in, is stored, or is read from the cell state â€” solving the vanishing gradient problem that plagues vanilla RNNs.
- Code walkthrough: Check that cell_state blends the previous cell state and new candidate memory using forget_gate and input_gate as weights, and that hidden_state is the cell state further filtered by output_gate.

**Learn more:**
- Website: [Dive into Deep Learning: Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RNNs+%2F+LSTMs+LSTM+gates+intuition+machine+learning+theory)

**Trade-offs:**
- Compared to a GRU's two gates, an LSTM's three gates (forget, input, output) give it a more expressive but also more parameter-heavy way to control what enters and leaves its memory cell.
- The extra gating machinery roughly triples the parameters and compute per cell compared to a vanilla RNN, so LSTMs are slower to train; GRUs offer a cheaper middle ground with fewer gates at a small cost in expressiveness.

**Practical software engineering use cases:**
- When to use it: Use an LSTM when you have enough data and compute budget to afford the extra parameters and want maximum control over what's retained.
- When not to use it: Don't default to a full LSTM on a small dataset or tight latency budget without trying a GRU first - the cheaper architecture is often just as good in practice.


In [ ]:
# RNNs / LSTMs - LSTM gates intuition
previous_cell = 0.8
candidate_memory = 0.4
forget_gate = 0.7
input_gate = 0.5
output_gate = 0.6
cell_state = forget_gate * previous_cell + input_gate * candidate_memory
hidden_state = output_gate * cell_state
print("cell_state:", round(cell_state, 3))
print("hidden_state:", round(hidden_state, 3))


### Checklist item: Text and time-series use cases

**Approach:**
- Protects deployment reliability: matching the architecture to the sequence type, whether short or long, univariate or multivariate, determines whether an RNN or LSTM is worth the added complexity over simpler baselines.
- Why this is the right approach: both text and time-series data share the same mathematical structure the RNN recurrence assumes, a sequence of observations where each element's meaning depends on its position and its predecessors - which is why the same recurrence equation, without modification, applies equally to a sequence of words or a sequence of numeric measurements.
- For this checklist item: LSTMs are well-suited to variable-length sequences (text, logs, time series) where long-range dependencies matter; they are slower than transformers on long sequences.
- Code walkthrough: Check that both examples get treated with the same 'next-step task uses order=True' flag - the same sequential-modeling logic applies whether the tokens are words or time-series values.

**Learn more:**
- Website: [Dive into Deep Learning: Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RNNs+%2F+LSTMs+Text+and+time-series+use+cases+machine+learning+theory)

**Trade-offs:**
- Unlike a Transformer, which needs a reasonably large dataset to learn positional relationships from scratch, an RNN's inductive bias toward sequence order can help it do more with less data on straightforward sequential tasks.
- For short sequences or where recent history dominates, simpler models such as lag features with a tree ensemble are often competitive and much cheaper to train and deploy than an LSTM.

**Practical software engineering use cases:**
- When to use it: Use an RNN/LSTM for a legacy system, a resource-constrained deployment, or a straightforward sequence task where a Transformer would be overkill.
- When not to use it: Don't build a new large-scale NLP system around an RNN today - the ecosystem, tooling, and pretrained models available for transformers make them the more practical default.


In [ ]:
# RNNs / LSTMs - Text and time-series use cases
examples = {
    "text": ["token_1", "token_2", "token_3"],
    "time_series": [100, 103, 107, 106],
}
for task, sequence in examples.items():
    print(task, "sequence_length=", len(sequence), "next-step task uses order=", True)


### Checklist item: Limitations compared to transformers

**Approach:**
- Protects model selection: knowing that RNNs process sequentially, which is slow to train and hard to parallelize, while transformers process in parallel via attention, which is fast to train and better at long-range dependencies, tells you which to default to for a new sequence task.
- Why this is the right approach: an RNN's recurrence requires h_(t-1) to be computed before h_t, mathematically forcing sequential, one-step-at-a-time computation - attention's definition computes a weighted sum over ALL positions simultaneously using only matrix multiplications, which have no such sequential dependency, precisely why attention parallelizes across a whole sequence while RNN computation structurally cannot.
- For this checklist item: RNNs process tokens sequentially and compress history into a hidden state, while transformers use attention to connect distant tokens directly and parallelize sequence processing.
- Code walkthrough: Check that max_rnn_path grows with sequence_length while max_attention_path stays fixed at 1 - that's the concrete number behind RNNs struggling with long-range dependencies compared to attention.

**Learn more:**
- Website: [Dive into Deep Learning: Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RNNs+%2F+LSTMs+Limitations+compared+to+transformers+machine+learning+theory)

**Trade-offs:**
- Unlike a Transformer's attention, which scales quadratically with sequence length but processes all positions in parallel, an RNN scales linearly but must process positions one at a time - opposite trade-offs in the same trade space.
- RNNs still have a place for streaming or online inference with strict memory constraints, since they use a constant-size hidden state versus attention's growing context, but for most offline NLP and sequence tasks today, transformers are the stronger default.

**Practical software engineering use cases:**
- When to use it: Use an RNN specifically where constant, bounded memory during inference matters more than raw accuracy, like an always-on streaming device.
- When not to use it: Don't choose an RNN over a transformer for a new offline NLP project purely out of familiarity - for most such tasks today, transformers are both more accurate and better supported.


In [ ]:
# RNNs / LSTMs - Limitations compared to transformers
sequence_length = 8
rnn_parallel_steps = sequence_length
transformer_parallel_steps = 1
max_rnn_path = sequence_length - 1
max_attention_path = 1
print("RNN sequential steps:", rnn_parallel_steps)
print("Transformer parallel steps per layer:", transformer_parallel_steps)
print("Longest dependency path RNN vs attention:", max_rnn_path, "vs", max_attention_path)


In [262]:
# Practice: RNNs / LSTMs

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Transformers

### Study checklist
- [ ] Attention intuition
- [ ] Self-attention
- [ ] Positional encoding
- [ ] Encoder and decoder architecture
- [ ] Why transformers changed NLP
- [ ] Basic transformer implementation idea

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Attention intuition

**Approach:**
- Protects representation: attention lets each position in a sequence directly weigh and combine information from every other position, removing the fixed-size bottleneck and sequential dependency that limits RNNs.
- Why this is the right approach: attention weights are computed via a softmax, a function mathematically guaranteed to produce values that are all positive and sum to exactly 1 - that guarantee is precisely what allows the weighted sum of values to be interpreted as a weighted AVERAGE, a mathematically well-defined blend rather than an arbitrary combination.
- For this checklist item: Represent tokens as vectors, compute how strongly each token should attend to others, then combine information using those weights.
- Code walkthrough: Check that attention_weights sum to 1.0, a softmax property, and that context is a weighted blend of values, pulled most toward whichever value has the highest attention weight.

**Learn more:**
- Website: [Dive into Deep Learning: Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Transformers+Attention+intuition+machine+learning+theory)

**Trade-offs:**
- Unlike an RNN's fixed-size hidden state, attention gives every position direct access to every other position's representation - a much richer but also much more compute-hungry mechanism.
- Computing attention between every pair of positions costs quadratically in sequence length, so very long sequences, such as long documents or high-resolution images as patches, become expensive without approximations like sparse or windowed attention.

**Practical software engineering use cases:**
- When to use it: Use attention-based architectures as your default for any new sequence modeling project where compute budget allows.
- When not to use it: Don't use full, dense attention on extremely long sequences, like entire books, without a sparse or windowed attention variant - plain attention's quadratic cost becomes prohibitive.


In [263]:
import math
# Transformers - Attention intuition
scores = [2.0, 1.0, 0.1]
exp_scores = [math.exp(score) for score in scores]
total = sum(exp_scores)
attention_weights = [score / total for score in exp_scores]
values = [10, 20, 30]
context = sum(weight * value for weight, value in zip(attention_weights, values))
print("attention:", [round(w, 3) for w in attention_weights])
print("context:", round(context, 3))


attention: [0.659, 0.242, 0.099]
context: 14.396


### Checklist item: Self-attention

**Approach:**
- Protects representation: self-attention lets a token's representation be updated based on other tokens in the same sequence, which is what lets the model resolve context-dependent meaning, such as what a pronoun refers to.
- Why this is the right approach: the attention score between two tokens is computed as their query-key dot product, which, per the same geometric identity behind cosine similarity, is largest when the two vectors point in similar directions - the precise mathematical mechanism by which self-attention assigns higher weight to tokens whose learned representations are more aligned with what the current token is looking for.
- For this checklist item: Represent tokens as vectors, compute how strongly each token should attend to others, then combine information using those weights.
- Code walkthrough: Check that each token's printed weights differ - 'ML' should attend most to itself and 'like', whose embedding points in a similar direction, rather than uniformly to all tokens.

**Learn more:**
- Website: [Dive into Deep Learning: Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Transformers+Self-attention+machine+learning+theory)

**Trade-offs:**
- Unlike a convolution's fixed local receptive field, self-attention can relate any two tokens directly regardless of their distance in the sequence - a structural advantage for capturing long-range dependencies.
- Multiple attention heads let the model attend to different types of relationships in parallel, but more heads and larger models multiply compute and memory cost, so head count is a capacity and cost trade-off like any other hyperparameter.

**Practical software engineering use cases:**
- When to use it: Use multiple attention heads, not just one, whenever you want the model to capture several different kinds of relationships between tokens simultaneously.
- When not to use it: Don't scale up head count or model size purely by default - verify each addition actually improves validation performance, since more heads also multiply compute and memory cost.


In [ ]:
# Transformers - Self-attention
import math

tokens = ["I", "like", "ML"]
embeddings = {
    "I": [1.0, 0.0],
    "like": [0.3, 0.7],
    "ML": [0.0, 1.0],
}
queries = keys = values = [embeddings[token] for token in tokens]

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def softmax(scores):
    exp_scores = [math.exp(score) for score in scores]
    total = sum(exp_scores)
    return [score / total for score in exp_scores]

for token, query in zip(tokens, queries):
    scores = [dot(query, key) for key in keys]
    weights = softmax(scores)
    context = [sum(weight * value[dim] for weight, value in zip(weights, values)) for dim in range(2)]
    print(token, "attends", [round(w, 3) for w in weights], "context", [round(v, 3) for v in context])


### Checklist item: Positional encoding

**Approach:**
- Protects representation: attention itself is permutation-invariant and has no inherent sense of order, so positional encoding is what injects word or token order back into the model.
- Why this is the right approach: the sinusoidal encoding uses sine and cosine at systematically varying frequencies across dimensions, a construction mathematically chosen so the encoding for any position can be expressed as a fixed LINEAR function of the encoding at any other position - that property is what lets the model learn to attend to relative positions using simple linear operations on the encodings.
- For this checklist item: Add position-dependent vectors to token embeddings so attention can distinguish the same token appearing at different sequence positions.
- Code walkthrough: Check that the encoding alternates between sin and cos values at different frequencies per dimension - that pattern is what lets the model distinguish position 2 from position 3 without an explicit position counter.

**Learn more:**
- Website: [Dive into Deep Learning: Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Transformers+Positional+encoding+machine+learning+theory)

**Trade-offs:**
- Unlike an RNN, which encodes order implicitly through its step-by-step processing, a transformer has to be given positional information explicitly - a requirement RNNs never had.
- Fixed sinusoidal encodings generalize to sequence lengths not seen in training without extra parameters, while learned positional embeddings can fit the training distribution better but may not extrapolate to longer sequences at inference time.

**Practical software engineering use cases:**
- When to use it: Use sinusoidal positional encodings when you expect to run inference on sequence lengths longer than anything seen in training.
- When not to use it: Don't use learned positional embeddings if extrapolation to longer sequences at inference time matters for your use case - they don't generalize past the lengths they were trained on.


In [ ]:
# Transformers - Positional encoding
import math

position = 2
d_model = 4
encoding = []
for i in range(d_model):
    angle = position / (10000 ** ((2 * (i // 2)) / d_model))
    encoding.append(math.sin(angle) if i % 2 == 0 else math.cos(angle))
print("position:", position)
print("positional encoding:", [round(v, 4) for v in encoding])


### Checklist item: Encoder and decoder architecture

**Approach:**
- Protects model selection: encoder-only models suit understanding tasks like classification or embeddings, decoder-only suits generation, and encoder-decoder suits sequence-to-sequence tasks like translation or summarization, so picking the wrong one wastes capacity on a training objective mismatched to your task.
- Why this is the right approach: masked self-attention sets the attention score to negative infinity for any position after the current one before the softmax - since softmax of negative infinity is exactly zero, this guarantees future tokens contribute nothing to the weighted sum, precisely the mathematical mechanism that enforces the decoder can't see tokens it hasn't generated yet.
- For this checklist item: The encoder builds contextual source representations; the decoder uses masked self-attention plus cross-attention to generate target tokens step by step.
- Code walkthrough: Check that encoder_memory is computed once from the source tokens, while next_token_context shows the decoder step reading both its own last token, masked self-attention, and that same fixed encoder memory, cross-attention.

**Learn more:**
- Website: [Dive into Deep Learning: Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Transformers+Encoder+and+decoder+architecture+machine+learning+theory)

**Trade-offs:**
- Compared to a decoder-only model, which reuses one stack for both understanding and generating text, an encoder-decoder architecture keeps the two roles separate - clearer for sequence-to-sequence tasks, at the cost of double the moving parts to train and serve.
- Encoder-decoder models are more complex to train and serve, needing two stacks and cross-attention, than a single decoder-only stack, which is part of why most modern large language models converged on the simpler decoder-only design.

**Practical software engineering use cases:**
- When to use it: Use an encoder-decoder architecture for genuine sequence-to-sequence tasks like translation or summarization with a fixed source and target.
- When not to use it: Don't reach for a full encoder-decoder setup for open-ended text generation or chat - that's exactly the use case decoder-only models were optimized for.


In [ ]:
# Transformers - Encoder and decoder architecture
encoder_tokens = ["source", "sentence"]
decoder_tokens = ["target"]
encoder_memory = [token.upper() for token in encoder_tokens]
next_token_context = {
    "masked_self_attention": decoder_tokens[-1],
    "cross_attention_reads": encoder_memory,
}
print("encoder memory:", encoder_memory)
print("decoder step context:", next_token_context)


### Checklist item: Why transformers changed NLP

**Approach:**
- Protects model selection: understanding that transformers solved RNNs' two biggest limits, parallelizable training and better long-range dependency modeling via direct attention, explains why they became the default architecture across NLP and beyond.
- Why this is the right approach: computing attention for all token pairs simultaneously is expressible as a small number of large matrix multiplications, operations hardware like GPUs are mathematically optimized to parallelize across many cores at once - that specific mathematical property let transformers scale to far larger models and datasets than an inherently sequential RNN computation could.
- For this checklist item: Transformers made long-context NLP more scalable by replacing strictly sequential recurrence with parallel token updates and direct attention between distant tokens.
- Code walkthrough: Check that transformer_attention_pairs grows quadratically, sequence_length squared, while parallel_token_updates stays equal to sequence_length - the quadratic-cost-but-parallel trade-off in one printed comparison.

**Learn more:**
- Website: [Dive into Deep Learning: Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Transformers+Why+transformers+changed+NLP+machine+learning+theory)

**Trade-offs:**
- Unlike RNNs, whose training time is bounded by sequential computation, transformers trade that bottleneck for one bounded by memory and quadratic attention cost - a different bottleneck, not the elimination of one.
- That same shift traded RNNs' cheap, constant-memory sequential inference for transformers' quadratic attention cost and much larger models, which is why serving large transformers is now its own engineering problem involving KV-caching, quantization, and batching.

**Practical software engineering use cases:**
- When to use it: Use transformer-based architectures as the default for any new NLP or sequence project today, given the maturity of pretrained models and tooling.
- When not to use it: Don't assume a transformer is automatically the right choice for every sequence problem regardless of scale - a small, latency-critical embedded system may still be better served by a lighter architecture.


In [ ]:
# Transformers - Why transformers changed NLP
sequence_length = 8
rnn_steps = sequence_length
transformer_attention_pairs = sequence_length * sequence_length
parallel_token_updates = sequence_length
print("RNN processes sequential steps:", rnn_steps)
print("Transformer compares token pairs with attention:", transformer_attention_pairs)
print("Transformer can update token representations in parallel:", parallel_token_updates)


### Checklist item: Basic transformer implementation idea

**Approach:**
- Protects debuggability: being able to sketch attention, positional encoding, and a feedforward block by hand on a toy example is what lets you debug shape mismatches and sanity-check a real implementation rather than trusting a library as a black box.
- Why this is the right approach: each described step, embedding lookup, adding positional vectors, computing attention scores via dot products, is a specific, well-defined mathematical operation with a precise formula - implementing them by hand on a tiny example is possible precisely because none of these steps involves anything beyond arithmetic already covered, applied in a specific sequence.
- For this checklist item: A transformer block combines token embeddings, positional information, self-attention, feed-forward layers, residual connections, and normalization.
- Code walkthrough: Check that with_position is computed by adding the positional vector to each token's embedding element-wise, and that block_outputs lists the four canonical steps of one transformer block in order.

**Learn more:**
- Website: [Dive into Deep Learning: Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Transformers+Basic+transformer+implementation+idea+machine+learning+theory)

**Trade-offs:**
- Unlike calling a library's transformer encoder class directly, hand-implementing attention forces you to confront every shape and scaling detail explicitly - slower to write, but far more instructive.
- A hand-rolled toy implementation is invaluable for building intuition but is not something to use in production; always use a well-tested library implementation for anything beyond learning, since subtle bugs in attention masking or scaling are easy to introduce and hard to spot.

**Practical software engineering use cases:**
- When to use it: Use a hand-rolled toy implementation specifically as a learning exercise, run on a tiny sequence you can trace by hand.
- When not to use it: Don't hand-roll attention for any real project - use a well-tested library implementation, since subtle bugs in masking or scaling are easy to introduce and hard to catch through normal testing.


In [ ]:
# Transformers - Basic transformer implementation idea
tokens = ["I", "like", "ML"]
embeddings = {"I": [1, 0], "like": [0, 1], "ML": [1, 1]}
positional = [[0.0, 0.1], [0.1, 0.0], [0.2, 0.1]]
with_position = [[a + b for a, b in zip(embeddings[token], pos)] for token, pos in zip(tokens, positional)]
block_outputs = ["embedding + position", "self-attention", "feed-forward", "residual + norm"]
print("input vectors:", with_position)
print("transformer block steps:", block_outputs)


In [269]:
# Practice: Transformers

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## NLP Basics

### Study checklist
- [ ] Tokenization
- [ ] Bag of words
- [ ] TF-IDF
- [ ] Text classification
- [ ] Named entity recognition
- [ ] Text preprocessing

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Tokenization

**Approach:**
- Protects representation: tokenization is the first step that converts raw text into discrete units a model can consume, so its granularity, whether word, subword, or character, directly determines vocabulary size and how the model handles rare or unseen words.
- Why this is the right approach: a model can only ever operate on a FIXED, finite vocabulary of known tokens, since each token maps to a specific numeric ID and embedding vector - subword tokenization mathematically guarantees any input string can be represented using this fixed vocabulary, falling back to smaller subword pieces down to individual characters if needed, which is why it never produces a true unknown-token failure.
- For this checklist item: Split text into tokens (words, subwords) and map each to an integer ID so the model receives numerical input â€” tokenization choice affects vocabulary size and OOV handling.
- Code walkthrough: Check that punctuation is stripped and the text is lowercased before splitting on whitespace - that's the minimal tokenization pipeline before any subword logic would be added.

**Learn more:**
- Website: [spaCy: Linguistic Features](https://spacy.io/usage/linguistic-features)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=NLP+Basics+Tokenization+machine+learning+theory)

**Trade-offs:**
- Unlike a whitespace split, which is free but brittle, subword tokenization such as BPE requires training its own small model on a corpus - extra setup that pays off in handling rare and out-of-vocabulary words gracefully.
- Subword tokenization such as BPE handles out-of-vocabulary words gracefully and keeps vocabulary size manageable, but it can split a word in ways that obscure its meaning, and changing the tokenizer after training silently breaks train/serve consistency.

**Practical software engineering use cases:**
- When to use it: Use the same tokenizer that a pretrained model was trained with whenever you fine-tune or serve that model - never swap it out.
- When not to use it: Don't roll your own simple tokenizer, like splitting on whitespace or punctuation, when working with a pretrained transformer - it expects the exact subword vocabulary it was trained on.


In [ ]:
# NLP Basics - Tokenization
text = "ML models learn from text."
tokens = text.lower().replace(".", "").split()
print(tokens)


### Checklist item: Bag of words

**Approach:**
- Protects representation: bag of words represents a document purely by word counts, discarding order, which makes it fast and simple but limits it to tasks where word presence alone carries most of the signal.
- Why this is the right approach: a bag-of-words vector is defined purely as a count per vocabulary word, with no dimension representing word ORDER or ADJACENCY - 'not good' and 'good' produce vectors differing only in whether the 'not' dimension is nonzero, why the representation has no mathematical mechanism to encode that 'not' semantically negates the following word.
- For this checklist item: Bag-of-words counts how many times each vocabulary word appears in a document â€” it ignores word order but is a strong baseline for classification.
- Code walkthrough: Check that each document's bow dict counts every vocabulary word's occurrences, including zeros for words absent from that document.

**Learn more:**
- Website: [spaCy: Linguistic Features](https://spacy.io/usage/linguistic-features)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=NLP+Basics+Bag+of+words+machine+learning+theory)

**Trade-offs:**
- Unlike TF-IDF, which reweights word counts by how distinctive they are across documents, plain bag of words treats every word's raw count as equally meaningful - simpler, but blind to how uninformative common words are.
- It's simple, fast, and a strong baseline for topic-level classification, but it can't distinguish "not good" from "good" since it loses word order and negation, and it produces very high-dimensional sparse vectors as vocabulary grows.

**Practical software engineering use cases:**
- When to use it: Use bag of words as a fast, no-tuning-required baseline before reaching for TF-IDF or embeddings on a new text classification problem.
- When not to use it: Don't use it for any task where word order or negation changes meaning, like sentiment analysis with not-good vs. good - it can't distinguish the two.


In [ ]:
# NLP Basics - Bag of words
documents = ["ml models learn", "models use data"]
vocabulary = sorted(set(" ".join(documents).split()))
bow = []
for doc in documents:
    tokens = doc.split()
    bow.append({word: tokens.count(word) for word in vocabulary})
print(vocabulary)
print(bow)


### Checklist item: TF-IDF

**Approach:**
- Protects representation: TF-IDF downweights words that are common across all documents, like "the," and upweights words distinctive to a given document, which improves on raw counts for tasks like search and document similarity.
- Why this is the right approach: the IDF term is defined as log(total_documents / documents_containing_word), a formula that mathematically approaches zero as a word appears in more documents - that's precisely the mechanism, not a tuning choice, by which TF-IDF automatically downweights common words like 'the' toward contributing almost nothing to the final score.
- For this checklist item: TF-IDF weights term frequency by inverse document frequency â€” common words (the, is) get low weight; rare discriminative words get high weight.
- Code walkthrough: Check that a word appearing in more documents, like 'ml' appearing in 2 of 3, gets a lower idf multiplier than a word appearing in only one document - TF-IDF specifically discounts common words.

**Learn more:**
- Website: [spaCy: Linguistic Features](https://spacy.io/usage/linguistic-features)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=NLP+Basics+TF-IDF+machine+learning+theory)

**Trade-offs:**
- Compared to bag of words, TF-IDF adds a corpus-wide reweighting step that downweights common words like 'the' - a small amount of extra computation for a generally stronger representation on search and classification tasks.
- It still ignores word order and semantics, so synonyms get no credit for being related, making it outperformed by embedding-based representations whenever semantic similarity matters, but it remains a fast, interpretable, no-training-required baseline.

**Practical software engineering use cases:**
- When to use it: Use TF-IDF plus a linear classifier as your second baseline, after plain bag of words, before trying embeddings or a transformer.
- When not to use it: Don't use TF-IDF where synonym or paraphrase matching matters - car and automobile get zero credit for meaning the same thing.


In [ ]:
# NLP Basics - TF-IDF
import math

documents = ["ml models learn", "models use data", "ml uses data"]
vocabulary = sorted(set(" ".join(documents).split()))
for doc in documents:
    tokens = doc.split()
    tfidf = {}
    for word in vocabulary:
        tf = tokens.count(word) / len(tokens)
        docs_with_word = sum(word in other.split() for other in documents)
        idf = math.log(len(documents) / docs_with_word)
        tfidf[word] = round(tf * idf, 3)
    print(doc, tfidf)


### Checklist item: Text classification

**Approach:**
- Protects evaluation integrity: text classification is the task that ties tokenization, representation choice, and model choice together, so it's the natural place to validate that the whole pipeline works end to end.
- Why this is the right approach: a linear classifier's decision is a weighted sum of input features; when the input is a TF-IDF vector, each weight directly corresponds to one specific word's learned importance for the class - that direct, one-weight-per-word correspondence is exactly why this pipeline remains fully interpretable, unlike an embedding whose dimensions don't correspond to any single word.
- For this checklist item: Text classification maps a document to a label (sentiment, spam, topic); features can be BoW, TF-IDF, or embeddings; the model can be logistic regression to a transformer.
- Code walkthrough: Check that the score is just the count of positive words minus negative words found in each text, and that the label flips based purely on the sign of that score.

**Learn more:**
- Website: [spaCy: Linguistic Features](https://spacy.io/usage/linguistic-features)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=NLP+Basics+Text+classification+machine+learning+theory)

**Trade-offs:**
- Unlike an embedding- or transformer-based pipeline, which needs a pretrained model and more infrastructure, a TF-IDF-plus-linear-classifier pipeline can be trained end-to-end from scratch on nothing but your own labeled data.
- A simple TF-IDF plus linear classifier is often a surprisingly strong, fast, interpretable baseline; reach for embeddings or transformers only once you've confirmed the simple pipeline's ceiling isn't good enough.

**Practical software engineering use cases:**
- When to use it: Use the TF-IDF-plus-linear-model pipeline first on any new text classification task to establish a real baseline before investing in anything heavier.
- When not to use it: Don't jump straight to a transformer-based classifier for a small, simple text classification task - you may be adding infrastructure and latency for an accuracy gain the simpler pipeline already captures.


In [ ]:
# NLP Basics - Text classification
positive_words = {"great", "useful", "fast"}
negative_words = {"buggy", "slow", "bad"}
texts = ["great fast model", "slow buggy app"]
for text in texts:
    tokens = set(text.split())
    score = len(tokens & positive_words) - len(tokens & negative_words)
    label = "positive" if score > 0 else "negative"
    print(text, "->", label, "score=", score)


### Checklist item: Named entity recognition

**Approach:**
- Protects data quality and downstream representation: NER extracts structured entities such as names, dates, and organizations from unstructured text, which is often a prerequisite step for downstream tasks like information extraction or building a knowledge base.
- Why this is the right approach: NER is mathematically framed as a per-token classification problem, assigning each token a label from a fixed tag set - because it's trained via the same classification objective as any other classifier, its accuracy is bounded by how similar the test tokens' contexts are to the labeled training examples it learned that classification boundary from.
- For this checklist item: NER labels each token as a named entity type (PERSON, ORG, LOC); it requires sequence-level models (CRF, transformer with token classification head).
- Code walkthrough: Check that only tokens present in the known_entities dict get tagged, and each keeps the exact type (ORG, LOCATION, DATE) assigned to it.

**Learn more:**
- Website: [spaCy: Linguistic Features](https://spacy.io/usage/linguistic-features)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=NLP+Basics+Named+entity+recognition+machine+learning+theory)

**Trade-offs:**
- Unlike keyword or regex matching, which needs to be manually written per entity type, NER learns to generalize entity patterns from labeled examples - but that generalization is only as good as how similar your entities are to its training data.
- Off-the-shelf NER models generalize reasonably to common entity types but degrade on domain-specific entities, such as drug names or part numbers, without fine-tuning on labeled in-domain data.

**Practical software engineering use cases:**
- When to use it: Use an off-the-shelf NER model for common entity types like people, organizations, dates, and locations, as a quick way to extract structure from text.
- When not to use it: Don't expect an off-the-shelf NER model to reliably catch domain-specific entities, like drug names or internal product codes, without fine-tuning on labeled examples from your own domain.


In [ ]:
# NLP Basics - Named entity recognition
text = "OpenAI opened an office in Tokyo in 2025"
known_entities = {"OpenAI": "ORG", "Tokyo": "LOCATION", "2025": "DATE"}
entities = [(token, known_entities[token]) for token in text.split() if token in known_entities]
print(entities)


### Checklist item: Text preprocessing

**Approach:**
- Protects data quality and train/serve consistency: steps like lowercasing, stopword removal, and stemming or lemmatization reduce noise and vocabulary size, but they must be applied identically at training and inference time or predictions will silently degrade.
- Why this is the right approach: stemming and stopword removal are deterministic functions that map several different input strings to the same reduced output, mathematically reducing the vocabulary size a count-based model needs to estimate parameters for - that reduction benefits a count-based model, but destroys exactly the surface-form information a pretrained transformer's embeddings were trained to use.
- For this checklist item: Preprocessing steps: lowercase, remove HTML/punctuation, strip stopwords, apply stemming or lemmatization â€” each step reduces noise but may remove signal.
- Code walkthrough: Check that tokens includes 'and' and 'more' while cleaned has them removed - that's the stopword-filtering step in isolation.

**Learn more:**
- Website: [spaCy: Linguistic Features](https://spacy.io/usage/linguistic-features)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=NLP+Basics+Text+preprocessing+machine+learning+theory)

**Trade-offs:**
- Unlike a transformer pipeline, which is trained on natural, largely unprocessed text, a bag-of-words or TF-IDF pipeline benefits from aggressive preprocessing like stemming and stopword removal - the right amount of preprocessing depends entirely on which representation follows it.
- Aggressive preprocessing such as stemming or stopword removal helps simple bag-of-words or TF-IDF models but can hurt embedding-based or transformer models, which are trained on natural, unprocessed text and rely on the very words, casing, and punctuation that heavy preprocessing removes.

**Practical software engineering use cases:**
- When to use it: Use standard preprocessing (lowercasing, stopword removal, stemming/lemmatization) ahead of a bag-of-words or TF-IDF model.
- When not to use it: Don't apply that same aggressive preprocessing ahead of an embedding or transformer-based model - it strips away casing, punctuation, and word forms those models were trained to use.


In [ ]:
# NLP Basics - Text preprocessing
import re

text = "ML models, models, and more models!"
stopwords = {"and", "more"}
tokens = re.findall(r"[a-z]+", text.lower())
cleaned = [token for token in tokens if token not in stopwords]
print("tokens:", tokens)
print("cleaned:", cleaned)


In [276]:
# Practice: NLP Basics

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Embeddings

### Study checklist
- [ ] Word embeddings
- [ ] Sentence embeddings
- [ ] Semantic similarity
- [ ] Vector databases
- [ ] Embedding-based search
- [ ] Evaluation of embedding quality

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Word embeddings

**Approach:**
- Protects representation: word embeddings map words to dense vectors such that semantically similar words end up close together, capturing meaning that one-hot or count-based representations can't.
- Why this is the right approach: word embeddings are trained so words appearing in similar CONTEXTS end up with similar vectors, a training objective that mathematically guarantees semantically related words land close together in the resulting vector space - but a static embedding assigns exactly one fixed vector per word regardless of context, why it mathematically cannot represent two different meanings of the same spelled word separately.
- For this checklist item: Word embeddings (Word2Vec, GloVe) map words to dense vectors where semantic similarity correlates with cosine similarity â€” king - man + woman â‰ˆ queen.
- Code walkthrough: Check that cosine(king, queen) comes out much higher than cosine(king, car) - the vectors were deliberately placed close together for semantically related words.

**Learn more:**
- Website: [Dive into Deep Learning: NLP Pretraining (word embeddings, word2vec, BERT)](https://d2l.ai/chapter_natural-language-processing-pretraining/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Embeddings+Word+embeddings+machine+learning+theory)

**Trade-offs:**
- Unlike a one-hot encoding, where every word is equally and maximally distant from every other word, embeddings place semantically similar words close together - a representation upgrade that comes from training on large amounts of text rather than from any hand-crafted rule.
- Static embeddings like Word2Vec or GloVe give one fixed vector per word regardless of context, so they can't distinguish a river bank from a financial bank; contextual embeddings from transformers solve this at a higher compute cost.

**Practical software engineering use cases:**
- When to use it: Use pretrained word embeddings when you need lightweight semantic representations and don't have the budget to fine-tune a full transformer.
- When not to use it: Don't use static word embeddings for a task where the same word's meaning changes by context, like bank - a contextual embedding model handles that correctly where a static one cannot.


In [ ]:
# Embeddings - Word embeddings
import math

word_vectors = {
    "king": [0.8, 0.2, 0.6],
    "queen": [0.75, 0.25, 0.65],
    "car": [0.1, 0.9, 0.2],
}
def cosine(a, b):
    return sum(x*y for x, y in zip(a, b)) / math.sqrt(sum(x*x for x in a) * sum(y*y for y in b))
print("king vs queen:", round(cosine(word_vectors["king"], word_vectors["queen"]), 3))
print("king vs car:", round(cosine(word_vectors["king"], word_vectors["car"]), 3))


### Checklist item: Sentence embeddings

**Approach:**
- Protects representation: sentence embeddings compress a whole sentence's meaning into a single vector, which is what makes similarity search and clustering over sentences or short passages computationally feasible.
- Why this is the right approach: naively averaging word vectors is a valid operation, but averaging is COMMUTATIVE, order doesn't affect the result, which mathematically discards word order entirely - a dedicated sentence-embedding model is trained with an objective that explicitly rewards preserving order-and-context-dependent meaning, precisely why it captures more than a simple average can by construction.
- For this checklist item: Sentence embeddings average or pool word vectors into one vector per sentence; models like sentence-transformers produce better representations than simple averages.
- Code walkthrough: Check that sentence_embedding averages the word vectors dimension by dimension, and that 'ml models' and 'models deploy' get different embeddings since they share only one word.

**Learn more:**
- Website: [Dive into Deep Learning: NLP Pretraining (word embeddings, word2vec, BERT)](https://d2l.ai/chapter_natural-language-processing-pretraining/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Embeddings+Sentence+embeddings+machine+learning+theory)

**Trade-offs:**
- Unlike averaging word embeddings, which is free but loses word-order information, a dedicated sentence embedding model is trained specifically to produce a single vector representing a whole sentence's meaning - better quality, at the cost of needing its own model.
- Pooling word or token embeddings naively, for example by averaging, loses word-order information and can dilute the meaning of longer sentences; models specifically trained to produce sentence embeddings via contrastive objectives perform noticeably better on semantic similarity tasks.

**Practical software engineering use cases:**
- When to use it: Use a dedicated sentence embedding model, not naive word-vector averaging, for any semantic search or clustering system you're building for real use.
- When not to use it: Don't rely on naive averaging of word embeddings for sentences longer than a phrase - the dilution effect gets worse as sentence length grows.


In [ ]:
# Embeddings - Sentence embeddings
word_vectors = {
    "ml": [1.0, 0.2],
    "models": [0.8, 0.3],
    "deploy": [0.2, 1.0],
}
def sentence_embedding(sentence):
    vectors = [word_vectors[word] for word in sentence.split()]
    return [sum(values) / len(values) for values in zip(*vectors)]
print("ml models:", [round(v, 3) for v in sentence_embedding("ml models")])
print("models deploy:", [round(v, 3) for v in sentence_embedding("models deploy")])


### Checklist item: Semantic similarity

**Approach:**
- Protects evaluation integrity: measuring similarity via a distance or cosine metric in embedding space is only meaningful if the embedding model's training objective actually aligns with what "similar" means for your task.
- Why this is the right approach: cosine similarity is defined as the dot product of two vectors divided by the product of their magnitudes, which by that formula's geometry isolates the ANGLE between two vectors from their length - a precise, bounded mathematical measure of directional alignment, why it's the standard way to compare embeddings regardless of how differently 'long' any two vectors happen to be.
- For this checklist item: Cosine similarity between embedding vectors measures semantic closeness; it is scale-invariant (only direction matters) and is the standard metric for retrieval.
- Code walkthrough: Check that 'model serving' scores much higher against the 'ml deployment' query than 'cake recipe' does - cosine similarity correctly separating the topically related pair from the unrelated one.

**Learn more:**
- Website: [Dive into Deep Learning: NLP Pretraining (word embeddings, word2vec, BERT)](https://d2l.ai/chapter_natural-language-processing-pretraining/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Embeddings+Semantic+similarity+machine+learning+theory)

**Trade-offs:**
- Unlike exact string or keyword matching, cosine similarity over embeddings can find related content that shares no words at all - but that same flexibility means it can also over-match on topic while missing your actual intent.
- Cosine similarity is fast and standard, but a high similarity score doesn't guarantee the two texts are equivalent for your specific purpose, such as paraphrase vs. same-topic-but-different-claim, so validate the similarity threshold against labeled examples from your domain.

**Practical software engineering use cases:**
- When to use it: Use cosine similarity over embeddings for near-duplicate detection, semantic search, or clustering by topic.
- When not to use it: Don't use a raw similarity score as a pass/fail decision without validating the threshold on labeled examples from your own domain first.


In [ ]:
# Embeddings - Semantic similarity
import math

sentences = {
    "ml deployment": [0.9, 0.8, 0.1],
    "model serving": [0.85, 0.75, 0.05],
    "cake recipe": [0.05, 0.1, 0.95],
}
def cosine(a, b):
    return sum(x*y for x, y in zip(a, b)) / math.sqrt(sum(x*x for x in a) * sum(y*y for y in b))
query = sentences["ml deployment"]
for text, vector in sentences.items():
    print(text, round(cosine(query, vector), 3))


### Checklist item: Vector databases

**Approach:**
- Protects deployment reliability and retrieval quality: a vector database is what makes nearest-neighbor search over millions of embeddings fast enough for production via approximate nearest-neighbor indexing, rather than a linear scan.
- Why this is the right approach: an approximate nearest-neighbor index is built on a mathematically provable trade-off - organizing points into a searchable graph or tree structure instead of comparing against every point reduces search time from linear to roughly logarithmic, at the cost of occasionally missing the exact true nearest neighbor - a defined, tunable trade-off, not an arbitrary approximation.
- For this checklist item: A vector database stores dense embeddings and supports approximate nearest-neighbor (ANN) search at scale â€” the retrieval layer in RAG and semantic search.
- Code walkthrough: Check that doc2, built to resemble the ML-related query, ranks above doc3 and doc4, the cooking and sports topics, in the sorted cosine-similarity results.

**Learn more:**
- Website: [Dive into Deep Learning: NLP Pretraining (word embeddings, word2vec, BERT)](https://d2l.ai/chapter_natural-language-processing-pretraining/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Embeddings+Vector+databases+machine+learning+theory)

**Trade-offs:**
- Unlike a brute-force linear scan over embeddings, which is simple but doesn't scale, a vector database's approximate nearest-neighbor index trades a small amount of exactness for the speed needed at real production scale.
- Approximate nearest-neighbor search trades a small amount of retrieval accuracy for large gains in speed and memory; index parameters, such as HNSW's ef and M, need tuning against your actual recall requirements, not left at defaults.

**Practical software engineering use cases:**
- When to use it: Use a vector database once your embedding collection is large enough, or once query latency requirements demand it beyond a linear scan.
- When not to use it: Don't add a vector database for a small, static collection of a few hundred or thousand embeddings - a simple in-memory linear scan is simpler to operate and plenty fast at that scale.


In [280]:
# Embeddings - Vector databases
import math

def cosine(a, b):
    dot  = sum(x*y for x,y in zip(a,b))
    na   = math.sqrt(sum(x**2 for x in a))
    nb   = math.sqrt(sum(x**2 for x in b))
    return dot/(na*nb) if na*nb>0 else 0

# Simulated 4-dimensional document embeddings
docs = {
    "doc1":[0.9,0.1,0.2,0.0],   # ML topic
    "doc2":[0.8,0.2,0.1,0.1],   # ML/deployment topic
    "doc3":[0.1,0.9,0.0,0.2],   # cooking topic
    "doc4":[0.0,0.1,0.9,0.1],   # sports topic
}
query = [0.85,0.1,0.15,0.05]    # ML-related query

print("Nearest-neighbor search (cosine similarity):")
scores = sorted(((did, cosine(query,vec)) for did,vec in docs.items()), key=lambda x:-x[1])
for did, score in scores:
    print(f"  {did}: {score:.3f}  {'#'*int(score*30)}")
print(f"\nNearest neighbor: {scores[0][0]}")
print("A vector DB indexes these embeddings for ANN search at millions of docs.")


Nearest-neighbor search (cosine similarity):
  doc1: 0.997  #############################
  doc2: 0.989  #############################
  doc3: 0.229  ######
  doc4: 0.189  #####

Nearest neighbor: doc1
A vector DB indexes these embeddings for ANN search at millions of docs.


### Checklist item: Embedding-based search

**Approach:**
- Protects user value: embedding-based semantic search retrieves results based on meaning rather than exact keyword match, which is what lets it handle paraphrased or conceptually related queries that keyword search misses.
- Why this is the right approach: embedding-based search ranks documents by the SAME cosine-similarity formula used for semantic similarity generally - because that formula only measures VECTOR alignment, not exact token overlap, it can mathematically assign a high score to a paraphrase sharing no words with the query, exactly the behavior keyword matching cannot replicate.
- For this checklist item: Embed queries and documents in the same space; retrieve nearest-document neighbors by cosine similarity â€” the basis of semantic search and RAG retrieval.
- Code walkthrough: Check that doc1, closest to the query in vector space, ranks first in the sorted output, purely from cosine similarity, without any keyword overlap check.

**Learn more:**
- Website: [Dive into Deep Learning: NLP Pretraining (word embeddings, word2vec, BERT)](https://d2l.ai/chapter_natural-language-processing-pretraining/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Embeddings+Embedding-based+search+machine+learning+theory)

**Trade-offs:**
- Unlike keyword (BM25) search, which requires exact or near-exact term overlap, embedding-based search can surface results that are conceptually related but share no words - a real strength that also introduces a failure mode keyword search never had.
- It can also retrieve semantically similar but factually wrong or irrelevant results that a keyword match would have avoided, so many production systems combine embedding search with keyword or BM25 search, called hybrid retrieval, rather than relying on either alone.

**Practical software engineering use cases:**
- When to use it: Use embedding-based search when queries are likely to be phrased differently than the documents that answer them.
- When not to use it: Don't rely on embedding-based search alone for queries needing exact matches, like product codes or names - combine it with keyword search to catch what pure semantic search misses.


In [ ]:
# Embeddings - Embedding-based search
import math

def cosine(a, b):
    return sum(x*y for x, y in zip(a, b)) / math.sqrt(sum(x*x for x in a) * sum(y*y for y in b))
query = [0.9, 0.7, 0.0]
doc_vectors = {"doc1": [0.8, 0.6, 0.1], "doc2": [0.2, 0.9, 0.1], "doc3": [0.0, 0.1, 0.9]}
ranked = sorted(((doc, cosine(query, vector)) for doc, vector in doc_vectors.items()), key=lambda pair: pair[1], reverse=True)
print([(doc, round(score, 3)) for doc, score in ranked])


### Checklist item: Evaluation of embedding quality

**Approach:**
- Protects evaluation integrity: without evaluating embeddings on a concrete downstream task or benchmark, you can't tell whether an embedding model actually captures the notion of similarity your application needs.
- Why this is the right approach: precision@k, the fraction of the top k retrieved results that are actually relevant, is a directly computable function of a labeled relevant-set and a ranked retrieval list - a precise, verifiable measurement of retrieval quality on YOUR specific queries, unlike a generic benchmark score measuring the same formula on a completely different, unrelated set of queries.
- For this checklist item: Evaluate embeddings with intrinsic benchmarks (analogy, similarity pairs) and extrinsic downstream task performance (retrieval recall@K, classification accuracy).
- Code walkthrough: Check that precision_at_2 is computed by intersecting the top-2 retrieved neighbors with the expected_relevant set - a real, if tiny, retrieval-quality metric rather than just eyeballing the neighbor list.

**Learn more:**
- Website: [Dive into Deep Learning: NLP Pretraining (word embeddings, word2vec, BERT)](https://d2l.ai/chapter_natural-language-processing-pretraining/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Embeddings+Evaluation+of+embedding+quality+machine+learning+theory)

**Trade-offs:**
- Unlike evaluating a classifier, where you have direct labels to score against, evaluating embedding quality usually means testing on a downstream task or benchmark - an indirect measure that can hide domain-specific gaps.
- Generic embedding benchmarks such as MTEB indicate general quality but don't guarantee performance on your specific domain and query style, so validate on a held-out sample of your own queries and documents before committing to a model.

**Practical software engineering use cases:**
- When to use it: Use a downstream task or benchmark, like retrieval accuracy on your own queries, to compare candidate embedding models before committing to one.
- When not to use it: Don't trust a generic benchmark like MTEB alone to predict performance on your specific domain and query style - validate on a held-out sample of your own data too.


In [ ]:
# Embeddings - Evaluation of embedding quality
neighbors = {
    "ml deployment": ["model serving", "production ml", "cake recipe"],
    "text embeddings": ["semantic search", "vector similarity", "database index"],
}
expected_relevant = {
    "ml deployment": {"model serving", "production ml"},
    "text embeddings": {"semantic search", "vector similarity"},
}
for query, retrieved in neighbors.items():
    precision_at_2 = len(set(retrieved[:2]) & expected_relevant[query]) / 2
    print(query, "precision@2=", precision_at_2)


In [283]:
# Practice: Embeddings

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## LLM Basics

### Study checklist
- [ ] Pretraining
- [ ] Fine-tuning
- [ ] Instruction tuning
- [ ] Context window
- [ ] Prompt engineering
- [ ] Limitations and hallucinations

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Pretraining

**Approach:**
- Protects representation: pretraining on a large, general text corpus, usually via next-token prediction, is what gives an LLM its broad language and world knowledge before any task-specific adaptation happens.
- Why this is the right approach: next-token prediction is mathematically the same cross-entropy classification objective used throughout this notebook, just with the 'class' being which token comes next out of the entire vocabulary - training on this single, well-defined objective at massive scale is what lets the model implicitly learn grammar, facts, and reasoning as a side effect of getting better at that one precise prediction task.
- For this checklist item: Pretraining optimizes next-token prediction on massive text corpora; the model learns grammar, facts, and reasoning patterns as a side effect of this objective.
- Code walkthrough: Check that predict_next('machine') returns 'learning' as the top completion - the bigram counts learned purely from counting word-pairs in the tiny corpus, a toy stand-in for next-token pretraining at scale.

**Learn more:**
- Website: [OpenAI: Model Optimization guide](https://developers.openai.com/api/docs/guides/model-optimization)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=LLM+Basics+Pretraining+machine+learning+theory)

**Trade-offs:**
- Unlike training a task-specific classifier from labeled data, pretraining learns from raw, mostly unlabeled text at a scale most teams will never reproduce - which is exactly why nearly everyone builds on an existing pretrained model instead.
- Pretraining is extremely compute- and data-intensive and is almost never done from scratch outside a few large labs; in practice you build on an existing pretrained model and focus effort on fine-tuning or prompting instead.

**Practical software engineering use cases:**
- When to use it: Use an existing pretrained model as your starting point for essentially any LLM-based project.
- When not to use it: Don't attempt to pretrain a language model from scratch for a typical product use case - the compute and data cost dwarfs the benefit versus starting from a pretrained checkpoint.


In [284]:
# LLM Basics - Pretraining
from collections import defaultdict

corpus = [
    "machine learning models learn from data",
    "deep learning models learn representations",
    "data science requires statistics and programming",
]
bigrams = defaultdict(lambda: defaultdict(int))
for sent in corpus:
    words = sent.split()
    for w1,w2 in zip(words, words[1:]):
        bigrams[w1][w2] += 1

def predict_next(word, n=3):
    if word not in bigrams: return []
    counts = bigrams[word]
    total  = sum(counts.values())
    return [(w, round(c/total,3)) for w,c in sorted(counts.items(),key=lambda x:-x[1])[:n]]

print("Bigram LM â€” tiny pretraining demo (objective: predict next token):")
for w in ["machine","learning","data","deep"]:
    print(f"  P(? | '{w}'): {predict_next(w)}")
print("\nReal LLMs train the same objective on trillions of tokens with transformers.")


Bigram LM â€” tiny pretraining demo (objective: predict next token):
  P(? | 'machine'): [('learning', 1.0)]
  P(? | 'learning'): [('models', 1.0)]
  P(? | 'data'): [('science', 1.0)]
  P(? | 'deep'): [('learning', 1.0)]

Real LLMs train the same objective on trillions of tokens with transformers.


### Checklist item: Fine-tuning

**Approach:**
- Protects model fit: fine-tuning adapts a pretrained model's weights to a specific task or domain using a smaller labeled dataset, which is how you specialize a general-purpose LLM without pretraining from scratch.
- Why this is the right approach: fine-tuning is mathematically the same gradient-descent weight-update procedure as training any neural network, just starting from pretrained weights instead of random initialization, using a smaller, task-specific dataset - starting from pretrained weights means optimization begins much closer to a good solution, precisely why it needs far less data and fewer steps than pretraining from scratch.
- For this checklist item: Fine-tuning continues training a pretrained LLM on a smaller labeled dataset for a specific task â€” far more data-efficient than training from scratch.
- Code walkthrough: Check that every training example ends with an assistant-role message, enforced by the assert, and that each one follows the same system/user/assistant structure a real fine-tuning dataset would need.

**Learn more:**
- Website: [OpenAI: Model Optimization guide](https://developers.openai.com/api/docs/guides/model-optimization)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=LLM+Basics+Fine-tuning+machine+learning+theory)

**Trade-offs:**
- Compared to prompting, which changes nothing about the model's weights, fine-tuning actually updates them - a deeper, more durable form of customization that also costs meaningfully more to do and to maintain.
- Full fine-tuning can achieve the best task performance but requires significant compute and risks catastrophic forgetting of general capabilities; parameter-efficient methods like LoRA or adapters trade a bit of ceiling performance for dramatically lower cost and easier deployment of multiple fine-tuned variants.

**Practical software engineering use cases:**
- When to use it: Use fine-tuning when you need consistent behavior on a narrow, well-defined task at high volume, and prompting alone hasn't reached the quality bar.
- When not to use it: Don't fine-tune before you've exhausted what prompt engineering can do - fine-tuning is more expensive to iterate on and easier to get wrong than adjusting a prompt.


In [285]:
# LLM Basics - Fine-tuning
training_examples = [
    {
        "messages": [
            {"role": "system", "content": "Answer as a concise ML tutor."},
            {"role": "user", "content": "What is overfitting?"},
            {"role": "assistant", "content": "Overfitting means a model memorizes training patterns that do not generalize."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Answer as a concise ML tutor."},
            {"role": "user", "content": "Why use validation data?"},
            {"role": "assistant", "content": "Validation data estimates generalization while you tune choices before final testing."},
        ]
    },
]

for example in training_examples:
    assert example["messages"][-1]["role"] == "assistant"

print("fine-tuning examples:", len(training_examples))
print("first prompt:", training_examples[0]["messages"][1]["content"])


fine-tuning examples: 2
first prompt: What is overfitting?


### Checklist item: Instruction tuning

**Approach:**
- Protects user value: instruction tuning trains a base language model on instruction-response pairs, which is what turns a raw next-token predictor into a model that actually follows user requests conversationally.
- Why this is the right approach: this is mathematically ordinary supervised fine-tuning, matching model outputs to target outputs, where the targets are specifically written as helpful responses to instructions - the model's weights are updated by the exact same gradient-based objective as any fine-tuning, so its resulting behavior is precisely as good as the instruction-response examples it was shown, no better and no worse.
- For this checklist item: Instruction tuning trains on (instruction, response) pairs so the model follows natural-language directives rather than just completing next tokens.
- Code walkthrough: Check that the instruction-tuned response answers the actual instruction (Bonjour) while the base-model completion just continues the prompt text, ignoring what was asked.

**Learn more:**
- Website: [OpenAI: Model Optimization guide](https://developers.openai.com/api/docs/guides/model-optimization)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=LLM+Basics+Instruction+tuning+machine+learning+theory)

**Trade-offs:**
- Unlike a raw pretrained model, which just continues text, an instruction-tuned model is specifically trained to follow a request - the difference between a model that completes an instruction as more text versus one that actually carries it out.
- It makes the model far more usable out of the box, but the quality and diversity of the instruction dataset directly caps the model's helpfulness and can introduce its own biases or refusal patterns that weren't present in the base model.

**Practical software engineering use cases:**
- When to use it: Use an instruction-tuned, chat model, not a raw base model, for essentially any user-facing application.
- When not to use it: Don't assume an instruction-tuned model's helpfulness generalizes evenly to every domain - its instruction dataset's coverage directly shapes where it's strong or weak.


In [ ]:
# LLM Basics - Instruction tuning

# A raw (base) language model just continues text - it has no built-in notion of
# "do what I ask". An instruction-tuned model is trained specifically on
# (instruction, response) pairs so it treats an instruction as a request to fulfill.

def base_model_continue(prompt):
    # A base model would just continue the text pattern, not answer the instruction.
    return prompt + " and then more text in the same style, ignoring the instruction."

def instruction_tuned_respond(prompt):
    # An instruction-tuned model recognizes the instruction and responds to it directly.
    responses = {
        "Translate 'hello' to French.": "Bonjour",
    }
    return responses.get(prompt, "I'll follow the instruction directly.")

prompt = "Translate 'hello' to French."
print("base model completion:      ", base_model_continue(prompt))
print("instruction-tuned response: ", instruction_tuned_respond(prompt))


### Checklist item: Context window

**Approach:**
- Protects representation and deployment reliability: the context window is the hard limit on how much text, including prompt, history, and retrieved documents, a model can attend to in a single call, so exceeding it silently truncates information the model needs.
- Why this is the right approach: self-attention's computational cost scales with the SQUARE of the sequence length, since it computes a score for every pair of tokens - that quadratic mathematical relationship is the direct, formula-level reason doubling the context length more than doubles the compute cost, and why there's a hard architectural limit on how many tokens can be attended to at once.
- For this checklist item: The context window is the maximum token count the model can attend to at once; longer context = higher cost and memory; chunking strategies manage it in RAG.
- Code walkthrough: Check that fit_to_context keeps the most recent messages and drops older ones once the 20-token budget is exceeded - check exactly which message got dropped.

**Learn more:**
- Website: [OpenAI: Model Optimization guide](https://developers.openai.com/api/docs/guides/model-optimization)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=LLM+Basics+Context+window+machine+learning+theory)

**Trade-offs:**
- Unlike RAM in a traditional program, which is cheap to over-provision just in case, every extra token in an LLM's context window costs money and latency on every single call - a cost model that rewards being deliberate about what you include.
- A larger context window lets you include more retrieved context or conversation history, but cost and latency typically scale with input length, and attention cost scales worse, so stuffing the context window isn't free even when it fits.

**Practical software engineering use cases:**
- When to use it: Use retrieval to pull in only the most relevant context for a given query, rather than stuffing everything potentially relevant into the prompt.
- When not to use it: Don't assume that because something fits in the context window, including it is free - cost and latency scale with input length, and very long contexts can dilute attention on what matters.


In [ ]:
# LLM Basics - Context window

# The context window is a hard limit on how many tokens (prompt + history +
# retrieved context) a model can attend to in one call. Anything beyond it must
# be dropped before the call is made.

MAX_CONTEXT_TOKENS = 20  # a tiny toy limit for illustration

conversation_history = ["Hello, I need help with my order.", "Sure, what's your order ID?",
                        "It's 48213.", "Let me look that up for you.", "Thanks!"]

def approx_tokens(text):
    return len(text.split())  # crude word-count stand-in for a real tokenizer

def fit_to_context(history, max_tokens):
    kept = []
    total = 0
    for msg in reversed(history):          # keep the most RECENT messages first
        cost = approx_tokens(msg)
        if total + cost > max_tokens:
            break
        kept.append(msg)
        total += cost
    return list(reversed(kept)), total

kept_history, used_tokens = fit_to_context(conversation_history, MAX_CONTEXT_TOKENS)
print(f"context window budget: {MAX_CONTEXT_TOKENS} tokens")
print(f"messages kept (most recent first, working backward): {kept_history}")
print(f"tokens used: {used_tokens}")
print(f"messages dropped: {len(conversation_history) - len(kept_history)}")


### Checklist item: Prompt engineering

**Approach:**
- Protects user value and evaluation integrity: how a task is phrased in the prompt directly affects output quality and consistency, so prompt design is itself a lever on model behavior, not just a way to ask nicely.
- Why this is the right approach: a language model's output is mathematically a probability distribution over next tokens CONDITIONED on the exact input sequence provided - changing the prompt changes the conditioning input to that same underlying function, precisely why different phrasing of the same request can shift the output distribution toward different completions, without any change to the model's weights.
- For this checklist item: Prompt engineering shapes model behavior through the system prompt, few-shot examples, chain-of-thought instructions, and output format constraints â€” without any weight updates.
- Code walkthrough: Check that the engineered prompt's quality score comes out higher than the vague prompt's - the improvement traces directly back to the specific phrasing rules the toy scoring function rewards.

**Learn more:**
- Website: [OpenAI: Model Optimization guide](https://developers.openai.com/api/docs/guides/model-optimization)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=LLM+Basics+Prompt+engineering+machine+learning+theory)

**Trade-offs:**
- Unlike fine-tuning, which requires training infrastructure and data, prompt engineering requires only text and iteration - the cheapest lever available for changing model behavior, though also the least durable one.
- Prompting is fast to iterate on and requires no training, but it's less reliable and harder to guarantee consistent behavior across edge cases than fine-tuning; for high-stakes or high-volume use cases, invest in evaluation sets to measure prompt changes quantitatively rather than eyeballing a few examples.

**Practical software engineering use cases:**
- When to use it: Use prompt engineering as your first and fastest lever for improving output quality, before considering RAG or fine-tuning.
- When not to use it: Don't rely on prompt engineering alone for a high-stakes or high-volume application without a real evaluation set - looking better on a few examples doesn't guarantee it's better on average.


In [ ]:
# LLM Basics - Prompt engineering

# How a task is phrased directly changes output quality and consistency, even
# though nothing about the underlying model changes between attempts.

def toy_model_response(prompt):
    # A stand-in for an LLM: rewards specific, structured prompts with a better
    # "response quality" score, and penalizes vague ones - a simplified illustration
    # of how prompt specificity affects real output quality.
    score = 0.3
    if "step by step" in prompt or "format" in prompt.lower():
        score += 0.3
    if "example" in prompt.lower():
        score += 0.2
    if len(prompt.split()) > 15:
        score += 0.2
    return min(score, 1.0)

vague_prompt = "Summarize this document."
engineered_prompt = ("Summarize this document in 3 bullet points, step by step, "
                     "following the format: [Point] - [Why it matters]. Example: "
                     "'Revenue grew 12% - driven by new product launches.'")

print(f"vague prompt quality score:      {toy_model_response(vague_prompt):.2f}")
print(f"engineered prompt quality score: {toy_model_response(engineered_prompt):.2f}")
print()
print("Same model, same task - the difference in output quality comes entirely from phrasing.")


### Checklist item: Limitations and hallucinations

**Approach:**
- Protects evaluation integrity and user trust: LLMs generate the statistically most plausible continuation, not a verified fact, so understanding this is what tells you where you must add grounding, such as retrieval, citations, or verification, rather than trusting raw output.
- Why this is the right approach: the model's training objective is defined purely as predicting the statistically most likely next token given its training data - that objective contains no mathematical term rewarding factual correctness specifically, so a fluent, high-probability continuation and a factually correct one are optimized identically whenever they differ, the root mathematical reason hallucination is a predictable consequence of the training objective.
- For this checklist item: LLMs hallucinate confidently, have a knowledge cutoff, and may leak training data; mitigate with grounding (RAG), constrained decoding, and output validation.
- Code walkthrough: Check that the ungrounded answer states a wrong year (1985) with just as much confidence as the grounded answer states the correct one (1998) - nothing in the ungrounded function's output signals it might be wrong.

**Learn more:**
- Website: [OpenAI: Model Optimization guide](https://developers.openai.com/api/docs/guides/model-optimization)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=LLM+Basics+Limitations+and+hallucinations+machine+learning+theory)

**Trade-offs:**
- Unlike a database query, which either returns real data or an explicit error, an LLM will confidently generate a plausible-sounding but false answer with no built-in signal that anything went wrong - a fundamentally different, and more dangerous, failure mode.
- Techniques like retrieval augmentation and lower sampling temperature reduce hallucination rates but don't eliminate them, so any user-facing or high-stakes application needs a way to verify or flag uncertain outputs rather than treating the model as a ground-truth source.

**Practical software engineering use cases:**
- When to use it: Use grounding techniques (retrieval, citations, verification steps) for any application where a wrong-but-confident answer would cause real harm.
- When not to use it: Don't deploy a raw LLM as a source of factual claims in a high-stakes domain like medical, legal, or financial without a verification or fallback layer.


In [ ]:
# LLM Basics - Limitations and hallucinations

# An LLM predicts the most PLAUSIBLE next tokens, not verified facts. Without
# grounding, it can produce a fluent, confident answer that is simply wrong.

knowledge_base = {
    "founding_year_acme_corp": 1998,
}

def ungrounded_answer(question):
    # No real lookup - the model just generates something plausible-sounding.
    return "Acme Corp was founded in 1985."           # confidently stated, but WRONG

def grounded_answer(question, kb):
    # Retrieval-augmented: look the fact up before answering.
    year = kb.get("founding_year_acme_corp")
    return f"Acme Corp was founded in {year}." if year else "I don't have that information."

question = "When was Acme Corp founded?"
print("ungrounded (hallucination risk):", ungrounded_answer(question))
print("grounded (retrieval-augmented): ", grounded_answer(question, knowledge_base))
print("actual fact:                    ", knowledge_base["founding_year_acme_corp"])


In [290]:
# Practice: LLM Basics

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## RAG

### Study checklist
- [ ] Retrieval augmented generation concept
- [ ] Chunking
- [ ] Embedding documents
- [ ] Vector search
- [ ] Prompt assembly
- [ ] Evaluation and grounding

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Retrieval augmented generation concept

**Approach:**
- Protects evaluation integrity and user value: RAG grounds an LLM's response in retrieved documents rather than relying purely on parametric memory, which directly reduces hallucination and lets the model answer about information outside its training data.
- Why this is the right approach: conditioning generation on retrieved context, P(answer | question, context) instead of just P(answer | question), mathematically gives the model additional, up-to-date evidence to condition its next-token predictions on - since the model's output is always a function of its input, providing more relevant input directly shifts that output distribution toward more accurate completions.
- For this checklist item: RAG grounds LLM answers in retrieved documents: embed the query, retrieve top-k similar chunks, inject them as context in the prompt before generation.
- Code walkthrough: Check that draft_answer only contains information present in retrieved_context - nothing in the answer wasn't first retrieved.

**Learn more:**
- Website: [OpenAI: Retrieval guide](https://developers.openai.com/api/docs/guides/retrieval)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RAG+Retrieval+augmented+generation+concept+machine+learning+theory)

**Trade-offs:**
- Unlike fine-tuning, which bakes new knowledge into the model's weights, RAG keeps knowledge external and retrievable - easier to update since you just change the documents, but entirely dependent on retrieval quality at query time.
- Answer quality is now bottlenecked by retrieval quality; a great generator with poor retrieval still produces confidently wrong answers, so RAG systems need as much evaluation investment in the retrieval step as in the generation step.

**Practical software engineering use cases:**
- When to use it: Use RAG when your knowledge base changes frequently or is too large to bake into the model via fine-tuning.
- When not to use it: Don't reach for RAG for knowledge that's small, stable, and fits comfortably in a prompt or the model's own training - the retrieval infrastructure adds complexity you may not need.


In [ ]:
# RAG - Retrieval augmented generation concept
query = "How do I monitor model drift?"
retrieved_context = [
    "Track input feature distributions over time.",
    "Compare live prediction distributions with training baselines.",
]
draft_answer = "Monitor drift by tracking feature and prediction distributions against training baselines."
print("1 query:", query)
print("2 retrieved context:", retrieved_context)
print("3 grounded answer:", draft_answer)


### Checklist item: Chunking

**Approach:**
- Protects representation and retrieval quality: how documents are split into chunks determines what unit of text can be retrieved and passed to the model, so poor chunking can separate a question from the context it needs to be answered.
- Why this is the right approach: a vector search ranks whole chunks by their overall embedding similarity to the query - if a chunk mixes a relevant sentence with irrelevant ones, its embedding is mathematically an average-like blend of all of it, diluting the relevant portion's contribution to the score, precisely why chunk size trades keeping enough context against diluting a specific relevant passage's signal.
- For this checklist item: Split long documents into overlapping chunks of 200-500 tokens; smaller chunks improve retrieval precision, larger chunks preserve more context per retrieval.
- Code walkthrough: Check that consecutive chunks share the overlap amount (2 words) at their boundary - that repetition is what 'overlap' means in a chunking strategy.

**Learn more:**
- Website: [OpenAI: Retrieval guide](https://developers.openai.com/api/docs/guides/retrieval)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RAG+Chunking+machine+learning+theory)

**Trade-offs:**
- Unlike embedding a whole document as one vector, which loses fine-grained relevance, chunking trades that loss for the need to tune a new parameter (chunk size) that has no single correct answer across document types.
- Small chunks retrieve more precisely but can lose surrounding context; large chunks preserve context but dilute relevance and cost more tokens per retrieved item, so chunk size and overlap should be tuned against your actual documents and queries, not left at a default.

**Practical software engineering use cases:**
- When to use it: Use smaller chunks with some overlap for documents where relevant answers are typically localized to a sentence or paragraph.
- When not to use it: Don't use one fixed chunk size across wildly different document types, like short FAQs vs. long technical manuals, without testing - what works for one can hurt retrieval on the other.


In [ ]:
# RAG - Chunking
document = "RAG retrieves documents before generation. Chunking splits long documents into searchable passages. Overlap preserves context across boundaries."
words = document.split()
chunk_size = 8
overlap = 2
chunks = []
start = 0
while start < len(words):
    chunk = " ".join(words[start:start + chunk_size])
    chunks.append(chunk)
    start += chunk_size - overlap
for i, chunk in enumerate(chunks, start=1):
    print(f"chunk_{i}:", chunk)


### Checklist item: Embedding documents

**Approach:**
- Protects representation: embedding documents into the same vector space as queries is what makes semantic retrieval possible, so the embedding model must be consistent between indexing time and query time.
- Why this is the right approach: cosine similarity between a query and a document is only meaningful if both were mapped into the SAME embedding space by the SAME model - since a different model defines a mathematically different space with no guaranteed correspondence, comparing across them invalidates the entire similarity computation.
- For this checklist item: Convert each chunk to an embedding vector using an embedding model; store the vectors in a vector database so they can be searched at query time.
- Code walkthrough: Check that embed() returns one count per vocabulary word for every document, and that doc1 and doc3, which share the word 'documents', have a non-zero count in the same vocabulary position.

**Learn more:**
- Website: [OpenAI: Retrieval guide](https://developers.openai.com/api/docs/guides/retrieval)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RAG+Embedding+documents+machine+learning+theory)

**Trade-offs:**
- Unlike a keyword index, which can be updated incrementally at little cost, changing the embedding model after documents are indexed requires re-embedding the entire corpus - a much heavier migration cost.
- Re-embedding a large corpus is expensive, so changing the embedding model later requires reprocessing the whole index; pick and validate the embedding model before building the index, not after.

**Practical software engineering use cases:**
- When to use it: Use a well-validated, stable embedding model choice before building out your document index at scale.
- When not to use it: Don't swap embedding models on a whim once you have a large index in production - budget for a full re-embedding pass, or query and document vectors will no longer be comparable.


In [ ]:
# RAG - Embedding documents
vocabulary = ["rag", "retrieves", "documents", "generation", "chunking"]
documents = {
    "doc1": "rag retrieves documents",
    "doc2": "generation uses context",
    "doc3": "chunking splits documents",
}

def embed(text):
    tokens = text.lower().split()
    return [tokens.count(word) for word in vocabulary]

for doc_id, text in documents.items():
    print(doc_id, embed(text), text)


### Checklist item: Vector search

**Approach:**
- Protects retrieval quality: the search step determines which chunks are actually available to the generator, so a bug or misconfiguration here, such as the wrong similarity metric or a stale index, silently degrades every downstream answer.
- Why this is the right approach: an embedding model is trained to place semantically similar text close together, but nothing in that objective guarantees an exact string like a product code maps to a distinctive location - a keyword search instead directly scores exact term overlap by formula, a different, complementary mathematical criterion hybrid search combines with embedding similarity for a fuller notion of relevance.
- For this checklist item: Encode the query as an embedding, compute cosine similarity against all chunk embeddings, and return the top-k most similar chunks as context for the LLM.
- Code walkthrough: Check that c2 and c4, the RAG and vector-DB-related chunks, score higher against query_emb than c1 and c3, unrelated ML topics - cosine similarity correctly surfacing the topically relevant chunks.

**Learn more:**
- Website: [OpenAI: Retrieval guide](https://developers.openai.com/api/docs/guides/retrieval)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RAG+Vector+search+machine+learning+theory)

**Trade-offs:**
- Unlike a lexical (BM25) index, which is excellent at exact term matches but blind to paraphrasing, vector search excels at paraphrase and concept matching but can miss exact identifiers like product codes - the two approaches fail in different, complementary ways.
- Pure vector search misses exact keyword or entity matches that a lexical search would catch, such as product codes or names; hybrid search combining vector and keyword retrieval is often more robust than vector search alone.

**Practical software engineering use cases:**
- When to use it: Use hybrid search (vector plus keyword) for production RAG systems where both conceptual queries and exact-match queries (names, codes) are expected.
- When not to use it: Don't rely on vector search alone if your users frequently search for exact identifiers or rare proper nouns - pure semantic search can under-retrieve those.


In [294]:
# RAG - Vector search
import math

def cosine(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na  = math.sqrt(sum(x**2 for x in a)); nb = math.sqrt(sum(x**2 for x in b))
    return dot/(na*nb) if na*nb>0 else 0

chunks = [
    {"id":"c1","text":"Gradient descent minimizes the loss function",          "emb":[0.8,0.1,0.3,0.0]},
    {"id":"c2","text":"RAG retrieves relevant documents before generating",    "emb":[0.1,0.9,0.1,0.2]},
    {"id":"c3","text":"Transformers use self-attention for sequence modeling", "emb":[0.3,0.2,0.9,0.1]},
    {"id":"c4","text":"Vector databases support approximate nearest neighbor", "emb":[0.1,0.8,0.2,0.9]},
]
query_emb = [0.15,0.85,0.15,0.75]   # related to retrieval / vector DBs

results = sorted(
    [(c["id"],c["text"],cosine(query_emb,c["emb"])) for c in chunks],
    key=lambda x:-x[2]
)
print("Query: 'How does vector search work in RAG?'")
print("\nTop-k retrieved chunks:")
for rank,(cid,text,score) in enumerate(results,1):
    print(f"  {rank}. [{cid}] score={score:.3f}  {text}")
print(f"\nTop-2 chunks injected into the LLM prompt as retrieved context.")


Query: 'How does vector search work in RAG?'

Top-k retrieved chunks:
  1. [c4] score=0.991  Vector databases support approximate nearest neighbor
  2. [c2] score=0.879  RAG retrieves relevant documents before generating
  3. [c3] score=0.378  Transformers use self-attention for sequence modeling
  4. [c1] score=0.252  Gradient descent minimizes the loss function

Top-2 chunks injected into the LLM prompt as retrieved context.


### Checklist item: Prompt assembly

**Approach:**
- Protects user value and evaluation integrity: how retrieved chunks are formatted and inserted into the prompt, including ordering, instructions, and citations, affects whether the model actually uses them correctly rather than ignoring or misattributing them.
- Why this is the right approach: attention weights, computed via softmax over all input tokens, mean every prompt token mathematically competes for a share of the model's limited attention budget - how retrieved chunks are formatted and ordered changes which tokens are present and where, directly changing the attention computation's input and output, why prompt assembly measurably affects context usage.
- For this checklist item: Assemble the retrieved chunks and the user query into a prompt template; include clear instructions so the model grounds its answer in the provided context.
- Code walkthrough: Check that the assembled prompt lists each retrieved chunk with its id before the question - that structure is what lets the model cite exactly which chunk it used.

**Learn more:**
- Website: [OpenAI: Retrieval guide](https://developers.openai.com/api/docs/guides/retrieval)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RAG+Prompt+assembly+machine+learning+theory)

**Trade-offs:**
- Unlike simple string concatenation, careful prompt assembly (ordering, instructions, citations) is itself an evaluated design surface - small formatting choices measurably change whether the model actually uses the retrieved context correctly.
- Including more retrieved chunks increases the chance of covering the right information but also increases prompt length and cost and the risk of the model getting distracted by irrelevant chunks, so tune the number of retrieved chunks against measured answer quality, not intuition.

**Practical software engineering use cases:**
- When to use it: Use explicit instructions and clear formatting, like numbered sources, when assembling retrieved chunks into a prompt, and treat that formatting as something to A/B test.
- When not to use it: Don't keep adding more retrieved chunks assuming more context can only help - past a point it increases cost and the chance the model gets distracted by irrelevant chunks.


In [ ]:
# RAG - Prompt assembly
question = "What does chunking do in RAG?"
retrieved_chunks = [
    {"id": "c2", "text": "Chunking splits long documents into searchable passages."},
    {"id": "c3", "text": "Overlap preserves context across chunk boundaries."},
]
prompt_lines = ["Answer using only the context. Cite chunk ids.", "", "Context:"]
for chunk in retrieved_chunks:
    prompt_lines.append(f"[{chunk['id']}] {chunk['text']}")
prompt_lines += ["", f"Question: {question}", "Answer:"]
prompt = "\n".join(prompt_lines)
print(prompt)


### Checklist item: Evaluation and grounding

**Approach:**
- Protects evaluation integrity: evaluating a RAG system means separately checking retrieval quality, whether the right documents were fetched, and generation faithfulness, whether the answer actually relied on them, since a good final answer can hide a broken retrieval step.
- Why this is the right approach: retrieval quality, was the relevant chunk retrieved at all, and generation faithfulness, did the answer's claims match the retrieved text, are two INDEPENDENT conditions that both must hold for a good final answer - since they're mathematically separate necessary conditions, measuring only the combined result can't tell you which one failed when the final answer is wrong.
- For this checklist item: Evaluate RAG with retrieval recall (did the right chunk appear in top-k?) and generation faithfulness (does the answer contradict any retrieved passage?).
- Code walkthrough: Check that the first two claims are marked grounded=True since they appear in supporting_chunks, while the third, 'RAG always guarantees factual answers', is grounded=False - a claim not backed by any retrieved text.

**Learn more:**
- Website: [OpenAI: Retrieval guide](https://developers.openai.com/api/docs/guides/retrieval)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=RAG+Evaluation+and+grounding+machine+learning+theory)

**Trade-offs:**
- Unlike evaluating a standalone LLM, evaluating a RAG system means separately scoring two different components, retrieval and generation - a more involved setup than a single end-to-end accuracy number.
- End-to-end answer-quality metrics are easiest to collect but conflate these two failure modes; investing in retrieval-specific metrics like recall@k and faithfulness or grounding checks costs more setup but makes failures far easier to diagnose and fix.

**Practical software engineering use cases:**
- When to use it: Use separate retrieval metrics (recall@k) and generation/faithfulness checks so you know which component to fix when overall answer quality drops.
- When not to use it: Don't rely solely on end-to-end answer-quality scores to debug a RAG system - they conflate two different failure modes and make root-causing a bad answer much harder.


In [ ]:
# RAG - Evaluation and grounding
answer = "Chunking splits long documents into searchable passages and overlap preserves context."
supporting_chunks = [
    "Chunking splits long documents into searchable passages.",
    "Overlap preserves context across chunk boundaries.",
]
claims = [
    "Chunking splits long documents into searchable passages",
    "overlap preserves context",
    "RAG always guarantees factual answers",
]
for claim in claims:
    grounded = any(claim.lower() in chunk.lower() for chunk in supporting_chunks)
    print(claim, "grounded=", grounded)


In [297]:
# Practice: RAG

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## AI Agents

### Study checklist
- [ ] Agent loop
- [ ] Tools and function calling
- [ ] Planning vs execution
- [ ] Memory
- [ ] Guardrails
- [ ] When agents are useful and when they are overkill

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Agent loop

**Approach:**
- Protects deployment reliability: the observe-think-act loop is what lets an agent take multiple steps toward a goal instead of producing a single one-shot response, so understanding it is understanding where the process can go wrong, such as a bad action, a bad observation, or an infinite loop.
- Why this is the right approach: framing the agent's behavior as observe-choose-act-repeat is mathematically the same Markov Decision Process structure used in reinforcement learning - defining explicit stopping conditions is required specifically because nothing in that loop's basic definition guarantees it terminates on its own without an external termination condition.
- For this checklist item: An agent loop runs Observe â†’ Think â†’ Act repeatedly until a stopping condition; define the observation format, available actions, and termination criteria upfront.
- Code walkthrough: Check that the tool_call dict, its observation, and the scratchpad entry get created in that order - observe the goal, act by calling the tool, record to scratchpad, then answer, the basic agent loop shape.

**Learn more:**
- Website: [OpenAI: Agents SDK](https://developers.openai.com/api/docs/guides/agents)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=AI+Agents+Agent+loop+machine+learning+theory)

**Trade-offs:**
- Unlike a single LLM call, which either succeeds or fails once, an agent loop can retry, gather more information, and adapt across multiple steps - powerful, but that same multi-step freedom is what needs explicit stopping conditions a single call never required.
- More loop iterations let the agent recover from mistakes and gather more information, but each iteration adds latency and cost and compounds the risk of drifting off-task, so agents need explicit stopping conditions and iteration limits.

**Practical software engineering use cases:**
- When to use it: Use an agent loop when a task genuinely requires multiple dependent steps that can't be planned out in advance.
- When not to use it: Don't wrap a task in an agent loop when a single well-crafted prompt already reliably solves it - the extra iterations only add latency and cost without benefit.


In [298]:
# AI Agents - Agent loop
def calculator(expression):
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError("unsafe expression")
    return eval(expression, {"__builtins__": {}}, {})

tools = {"calculator": calculator}
goal = "Find the weekly study hours for 2 hours/day over 6 days."

scratchpad = []
tool_call = {"tool": "calculator", "input": "2 * 6"}
observation = tools[tool_call["tool"]](tool_call["input"])
scratchpad.append((tool_call, observation))
answer = f"Study {observation} hours this week."

print("goal:", goal)
print("tool observation:", scratchpad[-1])
print("final answer:", answer)


goal: Find the weekly study hours for 2 hours/day over 6 days.
tool observation: ({'tool': 'calculator', 'input': '2 * 6'}, 12)
final answer: Study 12 hours this week.


### Checklist item: Tools and function calling

**Approach:**
- Protects deployment reliability and correctness: function calling lets an LLM take real, structured actions, such as querying a database or calling an API, instead of just generating text, which is what turns a chatbot into something that can actually do work.
- Why this is the right approach: a function call is mathematically just applying a KNOWN, defined function to a set of arguments - representing the model's intended action as a structured name-and-arguments pair, rather than free text, is what allows the exact same function to be called programmatically and deterministically, since a defined function requires its exact expected argument structure to execute correctly.
- For this checklist item: Tools are functions the agent can call (search, calculator, code runner); function calling lets the LLM emit a structured tool call that the harness executes.
- Code walkthrough: Check that tool_call is a structured dict with a name and arguments, and that lookup_price(**tool_call['arguments']) calls the real function using exactly those arguments - the structured-call-to-real-function link function calling relies on.

**Learn more:**
- Website: [OpenAI: Agents SDK](https://developers.openai.com/api/docs/guides/agents)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=AI+Agents+Tools+and+function+calling+machine+learning+theory)

**Trade-offs:**
- Unlike a chatbot that only produces text, function calling lets the model trigger real side effects such as database writes, API calls, or refunds - a qualitatively higher-stakes capability than anything a text-only model requires guarding.
- Giving a model more tools increases what it can accomplish but also expands the space of things it can get wrong or misuse, such as calling the wrong tool, malformed arguments, or unintended side effects, so tool access should be scoped and validated, not granted broadly by default.

**Practical software engineering use cases:**
- When to use it: Use function calling when the task genuinely requires taking an action in another system, not just generating a response.
- When not to use it: Don't grant a model broad, unscoped tool access just in case it's useful - scope each tool narrowly to what the specific task actually needs.


In [ ]:
# AI Agents - Tools and function calling
def lookup_price(item):
    prices = {"gpu": 500, "keyboard": 80}
    return prices[item]

tool_call = {"name": "lookup_price", "arguments": {"item": "gpu"}}
result = lookup_price(**tool_call["arguments"])
print("structured tool call:", tool_call)
print("tool result:", result)


### Checklist item: Planning vs execution

**Approach:**
- Protects correctness: separating deciding what to do from actually doing it makes an agent's behavior easier to inspect, constrain, and debug than a single step that both reasons and acts at once.
- Why this is the right approach: separating planning from execution means the plan exists as an explicit, INSPECTABLE object before any action with real side effects is taken - mathematically, this is the difference between generating a full solution path first versus taking one step and only then deciding the next, exactly what allows a plan to be checked or overridden before execution begins.
- For this checklist item: Planning produces a task decomposition before any action is taken; execution carries out each step and may re-plan on failure â€” separate them for reliability.
- Code walkthrough: Check that plan is built entirely before the execution loop runs, and that completed only gets populated afterward, one step at a time.

**Learn more:**
- Website: [OpenAI: Agents SDK](https://developers.openai.com/api/docs/guides/agents)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=AI+Agents+Planning+vs+execution+machine+learning+theory)

**Trade-offs:**
- Unlike a reactive agent that decides and acts in the same step, separating planning from execution adds a checkpoint you can inspect or override before anything happens - more overhead, but a real safety and debuggability gain for consequential actions.
- Explicit planning improves reliability on multi-step tasks but adds latency and complexity versus a simpler reactive agent; for short, well-defined tasks, a planning step can be pure overhead.

**Practical software engineering use cases:**
- When to use it: Use an explicit planning step for multi-step tasks with real-world consequences, where you want a chance to review the plan before execution.
- When not to use it: Don't add a separate planning step for short, low-stakes, single-action tasks - the extra round trip is pure latency with no real benefit there.


In [ ]:
# AI Agents - Planning vs execution
goal = "Prepare a model evaluation report"
plan = ["load metrics", "compare against baseline", "write summary"]
completed = []
for step in plan:
    completed.append({"step": step, "status": "done"})
print("goal:", goal)
print("plan:", plan)
print("execution log:", completed)


### Checklist item: Memory

**Approach:**
- Protects user value and correctness: giving an agent memory, whether of past turns, past actions, or retrieved facts, is what lets it maintain coherent, personalized behavior across a longer interaction instead of treating every step as stateless.
- Why this is the right approach: without persisted memory, each interaction is mathematically an independent function call with no shared state, unable to depend on anything from previous calls - adding memory changes the function's effective signature to include stored state, the specific structural change required for behavior to depend on anything beyond the immediate current input.
- For this checklist item: Memory can be short-term (conversation buffer), long-term (vector store retrieval), or episodic (past trajectories); choose based on how far back the agent must look.
- Code walkthrough: Check that relevant filters conversation_memory down to only entries whose type is 'preference' or 'project_context' - retrieval from memory, not just append-only storage.

**Learn more:**
- Website: [OpenAI: Agents SDK](https://developers.openai.com/api/docs/guides/agents)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=AI+Agents+Memory+machine+learning+theory)

**Trade-offs:**
- Unlike a single-turn LLM call, which is fully stateless, giving an agent memory means also taking on the systems-engineering problems that come with any persistent state: staleness, storage, and privacy.
- More persistent memory improves continuity but increases the risk of carrying forward stale or incorrect information, and raises privacy and data-retention concerns, so memory needs explicit scope, expiry, and update rules, not unlimited accumulation.

**Practical software engineering use cases:**
- When to use it: Use memory when an agent needs continuity across sessions, like remembering a user's stated preferences in a long-running assistant.
- When not to use it: Don't persist memory indefinitely without an expiry or update policy - stale or incorrect remembered facts can quietly corrupt future interactions.


In [ ]:
# AI Agents - Memory
conversation_memory = []
conversation_memory.append({"user": "I prefer short answers", "type": "preference"})
conversation_memory.append({"user": "My project is a churn model", "type": "project_context"})
relevant = [item for item in conversation_memory if item["type"] in {"preference", "project_context"}]
print("retrieved memory:", relevant)


### Checklist item: Guardrails

**Approach:**
- Protects deployment reliability and user safety: guardrails, such as input/output filtering, action allowlists, and human-in-the-loop checks, are what constrain an agent's behavior within acceptable bounds despite the underlying model's unpredictability.
- Why this is the right approach: checking a requested tool call against an explicit allowlist is a deterministic set-membership test that can be verified with total certainty - mathematically different from, and more reliable than, trying to infer 'is this action safe' from the model's own free-text reasoning, since set membership has no ambiguity while natural-language reasoning about safety can.
- For this checklist item: Guardrails validate or filter agent actions before execution (input validation, output classifiers, rate limits) to prevent unsafe, expensive, or off-policy actions.
- Code walkthrough: Check that 'send_email' gets blocked because it's not in allowed_tools, even though the request itself is well-formed - the guardrail check happens independently of whether the call would otherwise succeed.

**Learn more:**
- Website: [OpenAI: Agents SDK](https://developers.openai.com/api/docs/guides/agents)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=AI+Agents+Guardrails+machine+learning+theory)

**Trade-offs:**
- Unlike input validation in traditional software, which checks against a fixed schema, guardrails for an agent have to account for an open-ended range of model behaviors - a fundamentally fuzzier problem to bound.
- Stricter guardrails reduce the risk of harmful or costly mistakes but also reduce the agent's flexibility and can cause it to refuse or fail on legitimate edge cases, so guardrail strictness should match the real cost of the agent's mistakes in that specific use case.

**Practical software engineering use cases:**
- When to use it: Use stricter guardrails (approval steps, allowlists) in proportion to how costly or irreversible a mistake would be for a given action.
- When not to use it: Don't apply the same heavy guardrails to every action regardless of risk - over-restricting low-stakes actions causes legitimate requests to fail or need unnecessary human approval.


In [ ]:
# AI Agents - Guardrails
allowed_tools = {"calculator", "search_docs"}
requested_call = {"tool": "send_email", "input": "status update"}
if requested_call["tool"] not in allowed_tools:
    print("blocked tool call:", requested_call)
else:
    print("allowed:", requested_call)


### Checklist item: When agents are useful and when they are overkill

**Approach:**
- Protects deployment reliability: knowing agents' sweet spot, namely multi-step tasks needing tool use and adaptive decisions, prevents reaching for a complex agent loop where a single well-crafted prompt or a fixed pipeline would be simpler, cheaper, and more predictable.
- Why this is the right approach: an agent's expected total cost scales with the NUMBER OF LOOP ITERATIONS it takes, since each iteration is a full model call, while a fixed pipeline's cost is constant regardless of task complexity - for a task solvable in one step, that per-iteration cost formula makes an agent loop strictly more expensive for no accuracy benefit, the precise reason to prefer the simpler pipeline there.
- For this checklist item: Agents are worth the overhead when the task is dynamic, multi-step, and requires tool use; use a single LLM call (with a good prompt) for anything that can be answered in one shot.
- Code walkthrough: Check that use_agent is only True when steps>2 or tools>0 - a task needing just 1 step and 0 tools is explicitly marked not worth an agent.

**Learn more:**
- Website: [OpenAI: Agents SDK](https://developers.openai.com/api/docs/guides/agents)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=AI+Agents+When+agents+are+useful+and+when+they+are+overkill+machine+learning+theory)

**Trade-offs:**
- Unlike a fixed pipeline, whose behavior is fully predictable ahead of time, an agent trades that predictability for the ability to handle tasks whose steps can't be fully specified in advance.
- Agents handle open-ended, multi-step tasks that a fixed pipeline can't, but that flexibility comes with higher latency, cost, and unpredictability, so default to the simplest pipeline that solves the task and only add agentic loops when the task genuinely requires adaptive multi-step reasoning.

**Practical software engineering use cases:**
- When to use it: Use an agent specifically when the task's steps genuinely depend on intermediate results you can't predict ahead of time.
- When not to use it: Don't build an agent for a task with a known, fixed sequence of steps - a simple deterministic pipeline will be faster, cheaper, and more predictable.


In [ ]:
# AI Agents - When agents are useful and when they are overkill
tasks = {
    "summarize one paragraph": {"steps": 1, "tools": 0},
    "research, calculate, and file report": {"steps": 4, "tools": 2},
}
for task, traits in tasks.items():
    use_agent = traits["steps"] > 2 or traits["tools"] > 0
    print(task, "use_agent=", use_agent)


In [304]:
# Practice: AI Agents

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Generative Models

### Study checklist
- [ ] Generative vs discriminative models
- [ ] GANs (Generative Adversarial Networks)
- [ ] VAEs (Variational Autoencoders)
- [ ] Diffusion models
- [ ] Evaluating generative models
- [ ] Misuse and safety considerations

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Generative vs discriminative models

**Approach:**
- Protects representation and model selection: knowing whether a model learns P(label given features) or P(features given label) determines whether it can only classify, or can also create new, realistic data.
- Why this is the right approach: a discriminative model is mathematically defined as learning P(label | features) directly, while a generative model learns P(features | label) or P(features) itself - only the second formulation, having modeled the actual distribution of features, provides anything to SAMPLE FROM, the precise mathematical reason only generative models can create new data.
- For this checklist item: A discriminative model draws a boundary between classes; a generative model learns enough about each class's actual data distribution that it can sample brand-new examples from it.
- Code walkthrough: Check that discriminative_predict only ever returns a label for an existing point, while generate_new_point actually manufactures new coordinates that were never in the original data.

**Learn more:**
- Website: [Dive into Deep Learning: Generative Adversarial Networks](https://d2l.ai/chapter_generative-adversarial-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Generative+Models+Generative+vs+discriminative+models+machine+learning+theory)

**Trade-offs:**
- Compared to a discriminative model, a generative model has to learn a much richer picture of the data, which usually costs more training data and compute for the same classification accuracy.
- A generative model's ability to create new samples is exactly the extra capability a discriminative model lacks, which is why image, text, and audio generation are all built on generative approaches.

**Practical software engineering use cases:**
- When to use it: Use a discriminative model when the only thing you need is an accurate label or score for existing inputs, like spam detection.
- When not to use it: Don't reach for a generative model when all you need is classification accuracy - it solves a strictly harder problem (modeling the full data distribution) than you actually have.


In [ ]:
# Generative Models - Generative vs discriminative models
import random
random.seed(0)

class_a = [(x, x + random.uniform(-0.5, 0.5)) for x in range(5)]
class_b = [(x, x + 5 + random.uniform(-0.5, 0.5)) for x in range(5)]

def discriminative_predict(point):
    x, y = point
    return "A" if y < x + 2.5 else "B"

def generate_new_point(label):
    if label == "A":
        x = random.uniform(0, 4)
        return (x, x + random.uniform(-0.5, 0.5))
    x = random.uniform(0, 4)
    return (x, x + 5 + random.uniform(-0.5, 0.5))

test_point = (2, 3.0)
print("discriminative model just classifies:", discriminative_predict(test_point))
new_sample = generate_new_point("A")
print("generative model can also CREATE new data:", tuple(round(v, 2) for v in new_sample))


### Checklist item: GANs (Generative Adversarial Networks)

**Approach:**
- Protects representation and trainability: the adversarial setup, two networks with opposing objectives, is what lets a GAN learn to generate realistic data without ever being given an explicit density function to match.
- Why this is the right approach: GAN training is defined as a minimax game, the generator minimizes and the discriminator maximizes the same value function simultaneously - game theory shows this type of adversarial optimization has a theoretical equilibrium where the generator's output distribution exactly matches the real data distribution, the mathematical justification for why competition can in principle produce perfectly realistic samples.
- For this checklist item: The generator tries to produce fake data realistic enough to fool the discriminator, while the discriminator tries to correctly tell real data from the generator's fakes; both improve through this competition.
- Code walkthrough: Check that both generator_skill and discriminator_skill climb over the 5 printed rounds - each network is improving specifically in response to the other one getting better.

**Learn more:**
- Website: [Dive into Deep Learning: Generative Adversarial Networks](https://d2l.ai/chapter_generative-adversarial-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Generative+Models+GANs+%28Generative+Adversarial+Networks%29+machine+learning+theory)

**Trade-offs:**
- Compared to a VAE, GANs tend to produce sharper, more realistic samples, but the adversarial training process is less stable and harder to diagnose when it goes wrong.
- The competitive setup that makes GANs powerful is also what makes them hard to train: if the discriminator gets too strong too fast, the generator's gradients vanish and stop improving.

**Practical software engineering use cases:**
- When to use it: Use a GAN when you need to generate sharp, realistic samples, like images, and have enough data and compute to stabilize adversarial training.
- When not to use it: Don't use a GAN when you need a stable, easy-to-train model on a tight compute budget - adversarial training is notoriously prone to instability and mode collapse compared to other generative approaches.


In [ ]:
# Generative Models - GANs (Generative Adversarial Networks)
import random
random.seed(1)

real_data_quality = 0.9
generator_skill = 0.3
discriminator_skill = 0.5

print(f"{'round':>5}  {'generator':>10}  {'discriminator':>14}")
for round_num in range(1, 6):
    generator_skill += (discriminator_skill - generator_skill) * 0.3
    discriminator_skill += (real_data_quality - generator_skill) * 0.2
    print(f"{round_num:>5}  {generator_skill:>10.3f}  {discriminator_skill:>14.3f}")

print()
print("Both networks improve by competing against each other - training stops being useful")
print("once the generator's fakes are hard for the discriminator to tell apart from real data.")


### Checklist item: VAEs (Variational Autoencoders)

**Approach:**
- Protects representation: encoding inputs to a probability distribution instead of a single fixed point is what lets a VAE sample new, varied outputs from the latent space instead of only reconstructing exact inputs.
- Why this is the right approach: a VAE's loss function is mathematically derived as a lower bound on the true data likelihood, the ELBO, combining a reconstruction term with a term pushing the latent distribution toward a simple, known shape - that specific derivation is what guarantees sampling from that known latent shape and decoding produces a valid new sample.
- For this checklist item: A VAE's encoder outputs a mean and variance describing a distribution over the latent space; sampling from that distribution and decoding it produces either a reconstruction or, from a new random point, a brand-new generated output.
- Code walkthrough: Check that reconstructed differs slightly from original, since a random sample was drawn from the latent distribution rather than using its mean directly, and that a completely different new_z produces a genuinely new decoded output.

**Learn more:**
- Website: [Dive into Deep Learning: Generative Adversarial Networks](https://d2l.ai/chapter_generative-adversarial-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Generative+Models+VAEs+%28Variational+Autoencoders%29+machine+learning+theory)

**Trade-offs:**
- Compared to a plain autoencoder, a VAE trades perfect reconstruction for a smooth, sampleable latent space - that extra structure is what turns a compression tool into a generative one.
- The stochastic sampling step that enables generation also means a VAE's reconstruction of a specific input is never perfectly deterministic, unlike a standard autoencoder's.

**Practical software engineering use cases:**
- When to use it: Use a VAE when you want a smooth, well-structured latent space you can sample from and interpolate within, such as for controlled generation or anomaly detection.
- When not to use it: Don't use a VAE when you need the sharpest possible generated samples - VAEs are known to produce blurrier outputs than GANs or diffusion models on complex data like natural images.


In [ ]:
# Generative Models - VAEs (Variational Autoencoders)
import random
random.seed(2)

def encode(x):
    mean = sum(x) / len(x)
    variance = 0.1
    return mean, variance

def sample_latent(mean, variance):
    return mean + random.gauss(0, variance ** 0.5)

def decode(z):
    return [round(z + offset, 2) for offset in (-0.2, 0.0, 0.2)]

original = [1.0, 1.2, 0.9]
mean, variance = encode(original)
z = sample_latent(mean, variance)
reconstructed = decode(z)

print("original:     ", original)
print("latent mean:  ", round(mean, 3), " variance:", variance)
print("sampled point:", round(z, 3))
print("reconstructed:", reconstructed)

new_z = sample_latent(mean=0.5, variance=0.3)
print()
print("brand-new generated output:", decode(new_z))


### Checklist item: Diffusion models

**Approach:**
- Protects representation and trainability: framing generation as gradually reversing a noising process, rather than adversarial competition or a single latent sample, is what gives diffusion models their notably stable training and high sample quality.
- Why this is the right approach: the forward noising process is mathematically defined so that, after enough steps, the data provably converges to pure Gaussian noise regardless of its starting distribution - reversing that well-understood mathematical process step by step is a well-posed problem a network can be trained to approximate, precisely why diffusion models can start from noise and reliably work backward to a realistic sample.
- For this checklist item: The forward process incrementally adds noise to data until it's indistinguishable from pure noise; a trained model learns the reverse process, removing a bit of noise at each step to turn random noise into a realistic sample.
- Code walkthrough: Check that noisy_steps drifts further from the clean_signal at each forward step, and that the reverse denoise_step loop pulls the final noisy values back toward a smoother, less scattered result.

**Learn more:**
- Website: [Dive into Deep Learning: Generative Adversarial Networks](https://d2l.ai/chapter_generative-adversarial-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Generative+Models+Diffusion+models+machine+learning+theory)

**Trade-offs:**
- Compared to GANs, diffusion models are far more stable to train, since there's no adversarial competition to balance, but that stability costs many more sequential steps at generation time.
- Each denoising step only removes a small amount of noise by design, which is what makes the overall process stable, but also means generating one sample requires many repeated model calls instead of one.

**Practical software engineering use cases:**
- When to use it: Use a diffusion model when sample quality and training stability matter more than generation speed - they currently produce some of the highest-quality generated images available.
- When not to use it: Don't use a diffusion model when you need fast, low-latency generation - the many sequential denoising steps make inference notably slower than a single forward pass through a GAN or VAE.


In [ ]:
# Generative Models - Diffusion models
import random
random.seed(3)

clean_signal = [1.0, 2.0, 3.0, 4.0]

noisy_steps = [clean_signal]
for step in range(4):
    prev = noisy_steps[-1]
    noisy_steps.append([v + random.gauss(0, 0.5) for v in prev])

print("forward (noising) process:")
for step, values in enumerate(noisy_steps):
    print(f"  step {step}: {[round(v, 2) for v in values]}")

def denoise_step(values, amount=0.5):
    center = sum(values) / len(values)
    return [v + (center - v) * amount for v in values]

denoised = noisy_steps[-1]
for _ in range(4):
    denoised = denoise_step(denoised)
print()
print("reverse (denoising) process result:", [round(v, 2) for v in denoised])
print("(a real model learns to predict the noise at each step; this just approximates the idea)")


### Checklist item: Evaluating generative models

**Approach:**
- Protects evaluation integrity: a generative model has no single correct output to compare against, so evaluation has to compare the statistical properties of generated samples against real data instead.
- Why this is the right approach: FID is mathematically defined as a specific distance, the Frechet distance, between two Gaussian approximations of the real and generated feature distributions - because it's a proper mathematical distance, obeying the triangle inequality and equal to zero only when distributions match exactly, a lower score is provably, not just intuitively, evidence of a closer match to real data.
- For this checklist item: A distance metric between the real and generated distributions, like the mean and spread of each, tells you how close the generator's output distribution is to reality - the same idea real metrics like FID use on image feature vectors.
- Code walkthrough: Check that frechet_style_distance is much smaller between real_samples and good_generated than between real_samples and bad_generated - a lower distance directly reflects a better-matching generated distribution.

**Learn more:**
- Website: [Dive into Deep Learning: Generative Adversarial Networks](https://d2l.ai/chapter_generative-adversarial-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Generative+Models+Evaluating+generative+models+machine+learning+theory)

**Trade-offs:**
- Compared to a supervised model's accuracy on a labeled test set, evaluating a generative model is inherently less direct, since there's no single 'correct' output to check each generation against.
- A low distribution-distance score confirms the generated data statistically resembles real data in aggregate, but says nothing about individual sample quality or whether specific generations are usable.

**Practical software engineering use cases:**
- When to use it: Use a distribution-distance metric like FID (for images) whenever you need an automated, repeatable way to compare candidate generative models.
- When not to use it: Don't rely solely on a single distance number to judge generation quality for a human-facing product - pair it with actual human evaluation, since these metrics can miss issues like factual errors or subtle artifacts.


In [ ]:
# Generative Models - Evaluating generative models
def mean_and_std(values):
    m = sum(values) / len(values)
    s = (sum((v - m) ** 2 for v in values) / len(values)) ** 0.5
    return m, s

real_samples = [5.1, 5.3, 4.9, 5.0, 5.2]
good_generated = [5.0, 5.4, 4.8, 5.1, 5.3]
bad_generated = [1.0, 1.2, 0.9, 1.1, 1.0]

def frechet_style_distance(a, b):
    ma, sa = mean_and_std(a)
    mb, sb = mean_and_std(b)
    return (ma - mb) ** 2 + (sa - sb) ** 2

print("distance(real, good_generated):", round(frechet_style_distance(real_samples, good_generated), 3))
print("distance(real, bad_generated): ", round(frechet_style_distance(real_samples, bad_generated), 3))
print()
print("Lower distance means the generated distribution's mean/spread better matches real data -")
print("this is the same idea real FID (Frechet Inception Distance) uses for generated images.")


### Checklist item: Misuse and safety considerations

**Approach:**
- Protects deployment reliability and user safety: generative models can produce convincing fake images, audio, or text at scale, so a content-safety layer is a required system component, not an optional add-on.
- Why this is the right approach: a rule-based classifier applied to a request is a deterministic function with a verifiable decision boundary you can test and audit - relying on the generative model itself to decide not to produce harmful content means trusting the same statistical, next-token-likelihood mechanism responsible for hallucination, with no mathematical guarantee of respecting a boundary it wasn't explicitly trained to enforce.
- For this checklist item: Screening generation requests against a policy, such as blocking non-consensual likenesses or clearly harmful content, is what turns a raw generative capability into a responsibly deployed product feature.
- Code walkthrough: Check that only the requests with a real safety flag set get allowed=False - the benign painting request passes through untouched while the two problematic ones are blocked.

**Learn more:**
- Website: [Dive into Deep Learning: Generative Adversarial Networks](https://d2l.ai/chapter_generative-adversarial-networks/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Generative+Models+Misuse+and+safety+considerations+machine+learning+theory)

**Trade-offs:**
- Compared to a purely discriminative model, whose worst failure is usually a wrong label, a generative model's worst failure can be actively harmful synthetic content, which raises the bar for pre-deployment safeguards.
- A rule-based safety check like this one is fast and predictable but can't catch every creative attempt to bypass it, which is why production systems typically layer multiple detection methods together.

**Practical software engineering use cases:**
- When to use it: Use an explicit content-safety check on every generation request in any product exposing a generative model to real users.
- When not to use it: Don't treat safety filtering as something to bolt on after launch - build the check into the generation pipeline from the start, since retrofitting it after misuse occurs is far more costly.


In [ ]:
# Generative Models - Misuse and safety considerations
requests = [
    {"prompt": "a watercolor painting of a mountain lake", "flag": None},
    {"prompt": "a realistic photo of a real, named private individual without consent", "flag": "non-consensual likeness"},
    {"prompt": "step-by-step instructions to synthesize a dangerous chemical", "flag": "harmful content"},
]

def safety_check(request):
    return request["flag"] is None

for r in requests:
    allowed = safety_check(r)
    print(f"prompt: {r['prompt'][:55]!r:58s} allowed={allowed}")

print()
print("Generative models can produce convincing fake images, voices, and text at scale -")
print("a safety/content-policy layer is a required part of the system, not an afterthought.")


## Reinforcement Learning

### Study checklist
- [ ] RL problem setup (agent, environment, reward)
- [ ] Exploration vs exploitation
- [ ] Value-based methods (Q-learning)
- [ ] Policy-based methods
- [ ] Reward shaping and pitfalls
- [ ] When RL is (and isn't) the right tool

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: RL problem setup (agent, environment, reward)

**Approach:**
- Protects representation: framing a problem as agent, environment, and reward is what makes it solvable with RL at all - every RL algorithm assumes exactly this loop of taking an action, observing a new state, and receiving a reward signal.
- Why this is the right approach: this loop is mathematically formalized as a Markov Decision Process, a defined tuple of states, actions, transition probabilities, and rewards - framing a problem this way is what makes the large body of RL theory, convergence guarantees for Q-learning, policy gradient theorems, applicable at all, since those results are proven specifically for problems matching this exact structure.
- For this checklist item: The agent chooses actions, the environment responds with a new state and a reward, and the agent's only objective is to maximize the total reward it accumulates over time.
- Code walkthrough: Check that each call to env.step() returns a new state, a reward, and a done flag, and that the reward jumps to 10 only once the agent's position reaches the goal - the exact agent-environment-reward loop every RL method builds on.

**Learn more:**
- Website: [Dive into Deep Learning: Reinforcement Learning](https://d2l.ai/chapter_reinforcement-learning/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reinforcement+Learning+RL+problem+setup+%28agent%2C+environment%2C+reward%29+machine+learning+theory)

**Trade-offs:**
- Compared to supervised learning's fixed, labeled examples, RL only ever gets a reward signal from actions actually taken, so it has to learn from its own experience rather than a pre-collected dataset.
- Defining the environment and reward function correctly is entirely on you as the designer; a well-specified problem is straightforward to work with, but a poorly specified one produces an agent that optimizes the wrong thing.

**Practical software engineering use cases:**
- When to use it: Use this framing when you have a sequential decision-making problem where an agent's actions affect future states, and you can define a clear reward signal.
- When not to use it: Don't force this framing onto a one-shot prediction problem with no sequence of actions or state transitions - that's a standard supervised learning problem, not an RL one.


In [ ]:
# Reinforcement Learning - RL problem setup (agent, environment, reward)
class GridWorld:
    def __init__(self):
        self.position = 0
        self.goal = 4

    def step(self, action):
        self.position = max(0, min(self.goal, self.position + action))
        reward = 10 if self.position == self.goal else -1
        done = self.position == self.goal
        return self.position, reward, done

env = GridWorld()
state = env.position
print("agent starts at state:", state)
for step_num in range(6):
    action = 1
    state, reward, done = env.step(action)
    print(f"step {step_num}: action={action:+d} -> state={state}, reward={reward}, done={done}")
    if done:
        break


### Checklist item: Exploration vs exploitation

**Approach:**
- Protects model fit: an agent that only ever exploits its current best-known action can get permanently stuck on a mediocre option it discovered early, so deliberate exploration is what lets it discover better ones.
- Why this is the right approach: with a fixed number of trials, always exploiting the current best ESTIMATE can mathematically get permanently stuck if that estimate is wrong due to limited early samples - epsilon-greedy's guaranteed nonzero probability of trying any action ensures every action's estimate keeps improving, its estimation error provably driven toward zero as trials increase, formally preventing permanently converging on a suboptimal choice.
- For this checklist item: An epsilon-greedy strategy exploits the current best-estimated action most of the time, but occasionally explores a random action instead, purely to keep gathering information about options it hasn't tried enough.
- Code walkthrough: Check that the printed rounds show a mix of 'explore' and 'exploit' modes, and that arm_B, the true best option, ends up pulled far more often than the others once its estimated value rises.

**Learn more:**
- Website: [Dive into Deep Learning: Reinforcement Learning](https://d2l.ai/chapter_reinforcement-learning/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reinforcement+Learning+Exploration+vs+exploitation+machine+learning+theory)

**Trade-offs:**
- Compared to always exploiting the current best-known action, adding exploration costs some short-term reward on trials that don't pay off, but is what prevents the agent from missing a genuinely better option it just hasn't tried yet.
- The right amount of exploration (the epsilon value) is itself a trade-off: too little risks getting stuck on a suboptimal action, too much wastes reward on actions already known to be worse.

**Practical software engineering use cases:**
- When to use it: Use an explicit exploration strategy like epsilon-greedy whenever the agent starts with no reliable estimate of which actions are actually best.
- When not to use it: Don't set exploration too high for too long in a deployed system - excessive random exploration means real users or real resources are being spent on actions already known to be worse.


In [ ]:
# Reinforcement Learning - Exploration vs exploitation
import random
random.seed(1)

true_rewards = {"arm_A": 0.3, "arm_B": 0.7, "arm_C": 0.5}
estimated_values = {"arm_A": 0.5, "arm_B": 0.5, "arm_C": 0.5}
pulls = {"arm_A": 0, "arm_B": 0, "arm_C": 0}

epsilon = 0.2
for round_num in range(10):
    if random.random() < epsilon:
        arm = random.choice(list(estimated_values))
        mode = "explore"
    else:
        arm = max(estimated_values, key=estimated_values.get)
        mode = "exploit"
    reward = true_rewards[arm] + random.uniform(-0.1, 0.1)
    pulls[arm] += 1
    estimated_values[arm] += (reward - estimated_values[arm]) / pulls[arm]
    print(f"round {round_num}: {mode:>7} chose {arm}, reward={reward:.2f}")

print()
print("final estimated values:", {k: round(v, 2) for k, v in estimated_values.items()})
print("pulls per arm:", pulls)


### Checklist item: Value-based methods (Q-learning)

**Approach:**
- Protects model fit: learning a Q-value, the expected future reward of taking a specific action in a specific state, is what lets an agent choose good actions without ever needing an explicit model of the environment's dynamics.
- Why this is the right approach: the Q-learning update rule is derived directly from the Bellman equation, which mathematically states the optimal value of an action equals its immediate reward plus the discounted optimal value of the best action from the resulting state - Q-learning is proven to converge to the true optimal Q-values given enough visits to every state-action pair, a formal guarantee, not just an empirical tendency.
- For this checklist item: Q-learning updates its estimate for each (state, action) pair using the reward just received plus the best estimated value achievable from the next state, gradually propagating reward information backward through the state space.
- Code walkthrough: Check that the printed Q-values increase for move_right as state approaches the goal, and that state 3's move_right value (10.00) exactly matches the reward for reaching the goal directly.

**Learn more:**
- Website: [Dive into Deep Learning: Reinforcement Learning](https://d2l.ai/chapter_reinforcement-learning/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reinforcement+Learning+Value-based+methods+%28Q-learning%29+machine+learning+theory)

**Trade-offs:**
- Compared to a policy-based method, Q-learning indirectly derives its behavior from learned values rather than learning the action-selection rule directly, which can make it more sample-efficient but less natural for continuous action spaces.
- Q-learning can learn from experience generated by any policy, not just its own current one, which makes it more flexible for reusing past experience but also means it converges differently than on-policy methods.

**Practical software engineering use cases:**
- When to use it: Use Q-learning when the state and action space are small enough to track a value for every combination, and you don't have (or don't want to assume) a model of the environment's transition dynamics.
- When not to use it: Don't use tabular Q-learning on a problem with a huge or continuous state space - the table of Q-values becomes unmanageably large, which is exactly why deep Q-networks replace the table with a neural network.


In [ ]:
# Reinforcement Learning - Value-based methods (Q-learning)
import random
random.seed(0)

states = [0, 1, 2, 3, 4]
actions = [-1, 1]
Q = {(s, a): 0.0 for s in states for a in actions}
alpha, gamma = 0.5, 0.9

def step(s, a):
    s2 = max(0, min(4, s + a))
    reward = 10 if s2 == 4 else -1
    return s2, reward

for episode in range(200):
    s = 0
    for _ in range(20):
        a = random.choice(actions)
        s2, reward = step(s, a)
        best_next = max(Q[(s2, a2)] for a2 in actions)
        Q[(s, a)] += alpha * (reward + gamma * best_next - Q[(s, a)])
        s = s2
        if s == 4:
            break

print("learned Q-values (state, action) -> value:")
for s in states:
    print(f"  state {s}: move_left={Q[(s,-1)]:.2f}  move_right={Q[(s,1)]:.2f}")
print()
print("Q-values are highest for moving right near the goal - the agent learned the reward's shape.")


### Checklist item: Policy-based methods

**Approach:**
- Protects model fit: directly learning the action-selection rule, rather than first learning values and deriving actions from them, is what makes policy-based methods a natural fit for continuous or very large action spaces where Q-learning's table doesn't scale.
- Why this is the right approach: the policy gradient theorem provides an exact mathematical formula for how a policy's parameters should change to increase expected reward, without needing to first estimate a value function - that theorem justifies directly adjusting action probabilities based on observed reward, rather than an ad hoc rule, even though the resulting gradient estimate has high variance in practice.
- For this checklist item: A parameterized policy assigns a probability to each action; policy gradient methods nudge those parameters in the direction of whatever action distribution produced higher total reward, rather than estimating a value for every state-action pair.
- Code walkthrough: Check that policy_param generally trends upward across episodes as episodes with positive total_reward pull it toward a higher probability of moving right - the update rule is reinforcing whatever recently paid off.

**Learn more:**
- Website: [Dive into Deep Learning: Reinforcement Learning](https://d2l.ai/chapter_reinforcement-learning/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reinforcement+Learning+Policy-based+methods+machine+learning+theory)

**Trade-offs:**
- Compared to Q-learning's off-policy flexibility, policy-based methods typically learn on-policy, meaning they can only learn from experience generated by something close to their current policy, making them less sample-efficient.
- The raw policy-gradient update shown here has notoriously high variance episode to episode, which is why practical implementations add techniques like baselines or actor-critic methods to stabilize it.

**Practical software engineering use cases:**
- When to use it: Use a policy-based method when the action space is continuous or too large to enumerate, where a value table like Q-learning's isn't practical.
- When not to use it: Don't use a bare policy-gradient method as your first choice on a small, discrete-action problem where Q-learning would converge more reliably and with far less variance in the updates.


In [ ]:
# Reinforcement Learning - Policy-based methods
import random
random.seed(0)

policy_param = 0.6  # start slightly biased toward the goal direction

def sample_action(p):
    return 1 if random.random() < p else -1

def step(position, action):
    new_position = max(0, min(4, position + action))
    reward = 10 if new_position == 4 else -1
    return new_position, reward

for episode in range(6):
    position = 0
    total_reward = 0
    actions_taken = []
    for _ in range(15):
        action = sample_action(policy_param)
        actions_taken.append(action)
        position, reward = step(position, action)
        total_reward += reward
        if position == 4:
            break
    right_fraction = actions_taken.count(1) / len(actions_taken)
    policy_param += 0.1 * (right_fraction - policy_param) * (1 if total_reward > 0 else -1)
    policy_param = max(0.05, min(0.95, policy_param))
    print(f"episode {episode}: total_reward={total_reward:>4} policy_param(P(right))={policy_param:.3f}")


### Checklist item: Reward shaping and pitfalls

**Approach:**
- Protects evaluation integrity: the reward function is the only signal an RL agent optimizes, so a poorly designed one gets optimized just as faithfully as a well-designed one, often producing behavior nobody actually wanted.
- Why this is the right approach: an RL agent is mathematically guaranteed to converge, given enough training, toward maximizing exactly the reward function as WRITTEN, with no ability to distinguish that formula from the designer's true intent - any trajectory achieving a high value under the literal formula is an equally valid solution to the optimization as one that also achieves the intended real-world goal, precisely why a mismatched reward produces mismatched, if mathematically correct, behavior.
- For this checklist item: A reward that pays for merely surviving, regardless of progress toward the actual goal, can be maximized by an agent that never makes progress at all - a classic case of reward hacking.
- Code walkthrough: Check that badly_shaped_reward returns 1 for every step regardless of position, while better_shaped_reward correctly penalizes both distance from the goal and time taken - the lazy back-and-forth trajectory looks fine under the bad reward and clearly bad under the better one.

**Learn more:**
- Website: [Dive into Deep Learning: Reinforcement Learning](https://d2l.ai/chapter_reinforcement-learning/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reinforcement+Learning+Reward+shaping+and+pitfalls+machine+learning+theory)

**Trade-offs:**
- Compared to a sparse reward, given only on actually reaching the goal, a shaped reward gives the agent more frequent feedback and often trains faster, but shaping poorly can introduce exactly the kind of exploitable shortcut shown here.
- A reward that's too heavily shaped can inadvertently teach the agent to chase the shaping signal itself rather than the underlying goal it was meant to approximate.

**Practical software engineering use cases:**
- When to use it: Use a reward function that directly reflects progress toward the actual goal, and stress-test it by imagining what a lazy or degenerate policy could exploit.
- When not to use it: Don't ship a reward function without checking whether some trivial, non-progressing behavior could still score well under it - that's exactly the gap reward hacking exploits.


In [ ]:
# Reinforcement Learning - Reward shaping and pitfalls
def badly_shaped_reward(position, goal, steps_taken):
    return 1

def better_shaped_reward(position, goal, steps_taken):
    distance = abs(goal - position)
    return -distance - 0.1 * steps_taken

goal = 10
lazy_trajectory = [0, 1, 0, 1, 0, 1]
for step_num, pos in enumerate(lazy_trajectory):
    bad = badly_shaped_reward(pos, goal, step_num)
    good = better_shaped_reward(pos, goal, step_num)
    print(f"step {step_num}: position={pos}  bad_reward={bad}  better_reward={good:.1f}")

print()
print("The 'bad' reward happily pays the agent for surviving while going nowhere -")
print("a classic reward-hacking pitfall where the agent optimizes the reward, not the actual goal.")


### Checklist item: When RL is (and isn't) the right tool

**Approach:**
- Protects deployment reliability: knowing RL's sweet spot, sequential decisions with a way to try actions and observe outcomes, prevents reaching for a complex, sample-hungry RL setup where supervised learning would solve the same problem more simply.
- Why this is the right approach: RL's mathematical framework specifically models a SEQUENCE of dependent decisions where each action affects future states and rewards - for a one-shot prediction problem with no such dependency and labeled data already available, that same sequential machinery adds no mathematical benefit, since supervised learning's simpler objective is already the correct formal match for a one-shot labeled problem.
- For this checklist item: RL is worth its complexity specifically when decisions are sequential and a simulator or live environment lets the agent try actions and see results; for a one-shot prediction task with labeled data already available, supervised learning is simpler and cheaper.
- Code walkthrough: Check that use_RL is True only for the two problems with genuinely sequential decisions and a way to simulate or interact with the environment, while the two problems with existing labeled data and no real sequential structure get False.

**Learn more:**
- Website: [Dive into Deep Learning: Reinforcement Learning](https://d2l.ai/chapter_reinforcement-learning/index.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reinforcement+Learning+When+RL+is+%28and+isn%27t%29+the+right+tool+machine+learning+theory)

**Trade-offs:**
- Compared to supervised learning, RL typically needs vastly more interaction data to learn from, since it only receives a reward signal rather than a direct label for each decision.
- RL's ability to discover strategies no human explicitly labeled is a genuine advantage for problems like game-playing, but that same trial-and-error process is exactly why RL is a poor fit when real-world mistakes during training are costly or dangerous.

**Practical software engineering use cases:**
- When to use it: Use RL for problems like game-playing or robotics, where actions affect future states and you can practice via simulation or real interaction.
- When not to use it: Don't use RL for a problem you already have labeled examples for and that doesn't involve a sequence of dependent decisions - supervised learning will almost always be simpler, cheaper, and more sample-efficient there.


In [ ]:
# Reinforcement Learning - When RL is (and isn't) the right tool
problems = {
    "game-playing agent":      {"sequential_decisions": True,  "simulator_available": True,  "labeled_data": False},
    "product recommendation":  {"sequential_decisions": False, "simulator_available": False, "labeled_data": True},
    "robot navigation":        {"sequential_decisions": True,  "simulator_available": True,  "labeled_data": False},
    "email spam detection":    {"sequential_decisions": False, "simulator_available": False, "labeled_data": True},
}
for name, traits in problems.items():
    use_rl = traits["sequential_decisions"] and (traits["simulator_available"] or not traits["labeled_data"])
    print(f"{name:26s} sequential={traits['sequential_decisions']!s:6} simulator={traits['simulator_available']!s:6} use_RL={use_rl}")

print()
print("RL shines for sequential decision problems with a way to try actions and see outcomes -")
print("for a one-shot prediction problem with labeled data already available, supervised learning is simpler and cheaper.")


# Deployment and MLOps


## Model Deployment Basics

### Study checklist
- [ ] Saving models with joblib/pickle
- [ ] Batch inference
- [ ] REST API with FastAPI/Flask
- [ ] Input validation
- [ ] Model versioning
- [ ] Monitoring predictions

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Saving models with joblib/pickle

**Approach:**
- Protects deployment reliability: serializing a trained model is what lets you separate training, done once offline, from serving, done repeatedly online, and the serialization format determines what's portable and what's not.
- Why this is the right approach: pickling serializes an object's exact internal Python representation, its class, its attribute values, into a byte stream, and deserializing reconstructs that same representation - precisely why it requires the same class definitions in the loading environment, and why executing an untrusted pickle file can run arbitrary code: deserialization reconstructs and executes whatever object-construction instructions the byte stream contains.
- For this checklist item: Serialize the fitted pipeline (scaler + model together) with joblib or pickle; include version metadata so you know which artifact produced a prediction.
- Code walkthrough: Check that 'Original' and 'Loaded' predictions match exactly, and that loaded.version survives the pickle round-trip - serialization preserving both behavior and metadata.

**Learn more:**
- Website: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Model+Deployment+Basics+Saving+models+with+joblib%2Fpickle+machine+learning+theory)

**Trade-offs:**
- Unlike retraining a model from scratch every time you need it, serialization lets you save the trained state once - but that convenience is only as durable as the library versions and code used to create the file.
- Pickle and joblib are quick and simple but tie the saved model to the exact library versions and code used to train it, and are not safe to load from untrusted sources; production systems increasingly prefer safer or more portable formats, such as ONNX, once ad hoc scripts should become durable infrastructure.

**Practical software engineering use cases:**
- When to use it: Use joblib, or pickle, for quick internal deployments where you control both the training and serving environments closely.
- When not to use it: Don't use pickle/joblib to load a model file from an untrusted or external source - deserializing untrusted pickled data can execute arbitrary code.


In [227]:
# Model Deployment Basics - Saving models with joblib/pickle
import pickle, io

class SimpleModel:
    def __init__(self, weights, threshold=0.5):
        self.weights = weights; self.threshold = threshold; self.version = "1.0"
    def predict(self, features):
        score = sum(f*w for f,w in zip(features,self.weights))
        return int(score > self.threshold)

model = SimpleModel(weights=[0.3,-0.2,0.8])
print("Original:", model.predict([1.0, 0.5, 0.7]))

buf = io.BytesIO()
pickle.dump(model, buf); buf.seek(0)
loaded = pickle.load(buf)
print("Loaded  :", loaded.predict([1.0, 0.5, 0.7]))
print("Version :", loaded.version)
print("\nProduction: import joblib")
print("  joblib.dump(model,'model.pkl')  # serialize")
print("  model = joblib.load('model.pkl') # deserialize")


Original: 1
Loaded  : 1
Version : 1.0

Production: import joblib
  joblib.dump(model,'model.pkl')  # serialize
  model = joblib.load('model.pkl') # deserialize


### Checklist item: Batch inference

**Approach:**
- Protects deployment reliability and efficiency: running inference on data in bulk, rather than one request at a time, lets you fully utilize hardware throughput and amortize fixed overhead across many predictions.
- Why this is the right approach: processing many inputs through a matrix-multiplication-heavy model in one large batched call lets the hardware use vectorized instructions across the whole batch at once, the same mathematical principle behind NumPy vectorization being faster than a Python loop - precisely why throughput per prediction improves with batch size, up to hardware memory limits.
- For this checklist item: Run predictions on a full table or file in one shot â€” results written to a database or CSV; no latency requirement, optimized for throughput.
- Code walkthrough: Check that score_record is applied to every row in one pass, producing a label for all 5 records at once, rather than being called interactively per request.

**Learn more:**
- Website: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Model+Deployment+Basics+Batch+inference+machine+learning+theory)

**Trade-offs:**
- Unlike a real-time API, which must respond to each request individually, batch inference processes many requests together - much better hardware utilization, at the direct cost of not having a result until the whole batch finishes.
- Batch inference is efficient and simple to operate but introduces latency, since predictions aren't available until the batch runs, so it's only appropriate when near-real-time results aren't required; otherwise you need online or real-time serving instead.

**Practical software engineering use cases:**
- When to use it: Use batch inference for use cases like nightly scoring, reporting, or any workflow where results are needed on a schedule, not instantly.
- When not to use it: Don't use batch inference for a user-facing feature that needs an immediate response, like a live recommendation on page load - you need online/real-time serving instead.


In [228]:
# Model Deployment Basics - Batch inference
records = [
    {"user_id":101,"age":28,"score":0.7},
    {"user_id":102,"age":45,"score":0.3},
    {"user_id":103,"age":32,"score":0.8},
    {"user_id":104,"age":55,"score":0.2},
    {"user_id":105,"age":19,"score":0.9},
]

def score_record(r, threshold=0.5):
    prob = (r["score"] + r["age"]/100) / 2
    return {"user_id":r["user_id"], "prob":round(prob,3), "label":int(prob>threshold)}

results = [score_record(r) for r in records]
print(f"{'user_id':>8}  {'prob':>6}  {'label':>6}")
print("-" * 25)
for r in results:
    print(f"{r['user_id']:>8}  {r['prob']:>6.3f}  {r['label']:>6}")
print(f"\nPositive rate: {sum(r['label'] for r in results)/len(results):.0%}")
print("Results written to DB/CSV for downstream consumption.")


 user_id    prob   label
-------------------------
     101   0.490       0
     102   0.375       0
     103   0.560       1
     104   0.375       0
     105   0.545       1

Positive rate: 40%
Results written to DB/CSV for downstream consumption.


### Checklist item: REST API with FastAPI/Flask

**Approach:**
- Protects deployment reliability and user value: wrapping a model in a REST API is what makes it consumable by other services or applications in real time, turning a notebook artifact into an actual product dependency.
- Why this is the right approach: Pydantic's request model definition mathematically constrains the space of valid inputs to exactly the specified field names and types before any model code runs - a hard, checkable boundary, a request either matches the schema or is rejected, precisely why it can catch malformed requests deterministically, unlike relying on the model itself to somehow handle arbitrary invalid input gracefully.
- For this checklist item: Wrap the model in a FastAPI or Flask route that accepts JSON, validates input schema, calls model.predict(), and returns JSON â€” stateless per request.
- Code walkthrough: Check that the Pydantic PredictRequest model defines exactly the fields the /predict endpoint expects, and that the response combines a probability with a thresholded label - the request/response contract a real API would enforce.

**Learn more:**
- Website: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Model+Deployment+Basics+REST+API+with+FastAPI%2FFlask+machine+learning+theory)

**Trade-offs:**
- Unlike a batch scoring script that runs once and exits, a REST API must stay up, handle concurrent requests, and behave predictably under load - a genuinely different engineering problem than getting a model to produce correct predictions.
- A simple API is fast to stand up, but production readiness requires more than a working endpoint; input validation, timeouts, concurrency handling, and monitoring are what separate a demo API from one that survives real traffic.

**Practical software engineering use cases:**
- When to use it: Use a REST API when other services or applications need to request predictions synchronously, in real time.
- When not to use it: Don't treat a working FastAPI/Flask endpoint as production-ready by itself - without validation, timeouts, and monitoring, it's a demo, not a service that survives real traffic.


In [229]:
# Model Deployment Basics - REST API with FastAPI/Flask
SKELETON = '''
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class PredictRequest(BaseModel):
    age: float
    income: float
    days_inactive: int

class FakeModel:
    def predict_proba(self, features):
        score = 0.3 + 0.0001*features[1] - 0.01*features[2]
        return max(0.0, min(1.0, score))

model = FakeModel()

@app.post("/predict")
def predict(req: PredictRequest):
    prob = model.predict_proba([req.age, req.income, req.days_inactive])
    return {"probability": round(prob, 4), "label": int(prob > 0.5)}
'''
print(SKELETON)
print("Run: uvicorn app:app --reload")
print("Test: curl -X POST http://localhost:8000/predict \\")
print('      -H "Content-Type: application/json" \\')
print('      -d \'{"age":32,"income":60000,"days_inactive":5}\'')



from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class PredictRequest(BaseModel):
    age: float
    income: float
    days_inactive: int

class FakeModel:
    def predict_proba(self, features):
        score = 0.3 + 0.0001*features[1] - 0.01*features[2]
        return max(0.0, min(1.0, score))

model = FakeModel()

@app.post("/predict")
def predict(req: PredictRequest):
    prob = model.predict_proba([req.age, req.income, req.days_inactive])
    return {"probability": round(prob, 4), "label": int(prob > 0.5)}

Run: uvicorn app:app --reload
Test: curl -X POST http://localhost:8000/predict \
      -H "Content-Type: application/json" \
      -d '{"age":32,"income":60000,"days_inactive":5}'


### Checklist item: Input validation

**Approach:**
- Protects data quality and deployment reliability: validating a request's shape, types, and ranges before it reaches the model is what prevents malformed or out-of-distribution inputs from causing a crash or a silently wrong prediction.
- Why this is the right approach: a model's mathematical function, matrix multiplications and activations, is defined for ANY numeric input of the right shape, valid or not - there's no mechanism inside the model computation itself that distinguishes a sensible input from nonsense, precisely why validation has to happen as an explicit, separate check before the input ever reaches that computation.
- For this checklist item: Decide what future production data will look like, then split so validation simulates that future rather than leaking information across rows.
- Code walkthrough: Check that request passes validation only because every field in schema is present and correctly typed - try mentally changing 'age' to a string and see which line would catch it.

**Learn more:**
- Website: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Model+Deployment+Basics+Input+validation+machine+learning+theory)

**Trade-offs:**
- Unlike the model itself, which will happily produce a number for almost any input you give it, input validation is what actually decides whether that input made sense in the first place - a check the model has no way to perform on its own.
- Strict validation catches more bad inputs but can reject legitimate edge cases the schema didn't anticipate; validation rules should be derived from the training data's actual distribution and updated as that distribution evolves, not hardcoded once and forgotten.

**Practical software engineering use cases:**
- When to use it: Use schema-based validation (types, ranges, required fields) at the API boundary before any input reaches the model.
- When not to use it: Don't rely on the model to fail gracefully on bad input - an out-of-distribution input often produces a confident, silently wrong prediction rather than an error.


In [230]:
# Model Deployment Basics - Input validation
schema = {"age": int, "income": float, "country": str}
request = {"age": 34, "income": 72000.0, "country": "IN"}
errors = []
for field, expected_type in schema.items():
    if field not in request:
        errors.append(f"missing {field}")
    elif not isinstance(request[field], expected_type):
        errors.append(f"{field} must be {expected_type.__name__}")
print("valid request:", not errors)
print("errors:", errors)


valid request: True
errors: []


### Checklist item: Model versioning

**Approach:**
- Protects deployment reliability and reproducibility: versioning models, and the data and code that produced them, is what lets you trace a production prediction back to exactly which model made it, and roll back safely if a new version regresses.
- Why this is the right approach: a specific version's predictions are a deterministic function of that version's exact weights - overwriting those weights in place means the function that produced a past prediction no longer exists anywhere to be re-examined, precisely why explicit versioning, keeping every past weight-set retrievable, is required to ever explain or roll back a specific historical prediction.
- For this checklist item: Tag each serialized model with a version string, training date, and metric snapshot so you can roll back or audit any deployed prediction.
- Code walkthrough: Check that each registered version keeps its own metrics, feature list, and notes - enough information to know exactly what changed between v1.0, v1.1, and v2.0.

**Learn more:**
- Website: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Model+Deployment+Basics+Model+versioning+machine+learning+theory)

**Trade-offs:**
- Unlike code, which is naturally versioned by git, a trained model artifact needs its own explicit versioning scheme - a step teams new to ML deployment often skip until they need to roll back and can't.
- Versioning every model adds storage and process overhead, but skipping it means you can't reproduce past predictions or safely roll back a bad deployment; the cost is worth paying well before you think you need it, since incidents are when you actually need the history.

**Practical software engineering use cases:**
- When to use it: Use explicit model versioning, a version ID stored alongside the artifact and its training metadata, from your very first production deployment.
- When not to use it: Don't overwrite a previous model file in place when deploying a new version - you lose the ability to roll back or explain a past prediction.


In [231]:
# Model Deployment Basics - Model versioning
registry = []

def register(name, version, metrics, features, notes):
    registry.append({"name":name,"version":version,"metrics":metrics,
                      "features":features,"notes":notes})

register("churn_model","v1.0",{"accuracy":0.80,"recall":0.65},
         ["age","income"],"Baseline logistic regression")
register("churn_model","v1.1",{"accuracy":0.83,"recall":0.72},
         ["age","income","days_inactive"],"Added inactivity feature")
register("churn_model","v2.0",{"accuracy":0.87,"recall":0.80},
         ["age","income","days_inactive","support_tickets"],"Switched to GradientBoosting")

print("Model Registry:")
print("-" * 65)
for m in registry:
    print(f"  {m['name']} {m['version']}")
    print(f"    metrics : {m['metrics']}")
    print(f"    features: {m['features']}")
    print(f"    notes   : {m['notes']}")


Model Registry:
-----------------------------------------------------------------
  churn_model v1.0
    metrics : {'accuracy': 0.8, 'recall': 0.65}
    features: ['age', 'income']
    notes   : Baseline logistic regression
  churn_model v1.1
    metrics : {'accuracy': 0.83, 'recall': 0.72}
    features: ['age', 'income', 'days_inactive']
    notes   : Added inactivity feature
  churn_model v2.0
    metrics : {'accuracy': 0.87, 'recall': 0.8}
    features: ['age', 'income', 'days_inactive', 'support_tickets']
    notes   : Switched to GradientBoosting


### Checklist item: Monitoring predictions

**Approach:**
- Protects deployment reliability: monitoring prediction distributions and downstream outcomes in production is the only way to catch model degradation, such as drift, data quality issues, or bugs, that offline evaluation on historical data can't reveal.
- Why this is the right approach: a model's learned function is fixed at training time, but the real-world input distribution it receives in production can drift over time - since the model's mapping from input to output never updates itself, a shift in the input distribution mathematically changes the DISTRIBUTION OF OUTPUTS the fixed function produces, which monitoring is specifically designed to detect by comparing current output statistics against the training-time baseline.
- For this checklist item: Log prediction score distributions over time; alert when the positive rate or mean score drifts beyond baseline thresholds.
- Code walkthrough: Check that 'current's pos_rate is flagged YES for drift while mean or std may or may not be, since the shift was specifically injected into the distribution's center - the per-metric flags aren't all triggered by the same underlying change.

**Learn more:**
- Website: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Model+Deployment+Basics+Monitoring+predictions+machine+learning+theory)

**Trade-offs:**
- Unlike offline evaluation, which only sees historical data, production monitoring is the only way to catch a model quietly degrading against live, shifting data.
- Comprehensive monitoring, covering input drift, prediction drift, outcome tracking, and latency, costs engineering effort to build and maintain, but the alternative is discovering model failures only when a stakeholder complains; monitoring should be scoped to the failure modes that would actually matter for your use case.

**Practical software engineering use cases:**
- When to use it: Use monitoring on prediction distributions and, where available, real outcomes for any model whose input data is likely to drift over time.
- When not to use it: Don't skip monitoring for a model just because it performed well at launch - the launch-time evaluation says nothing about how it performs six months later against changed data.


In [232]:
# Model Deployment Basics - Monitoring predictions
import random

def window(n, shift=0.0, seed=0):
    random.seed(seed)
    return [round(random.gauss(0.4+shift, 0.15), 3) for _ in range(n)]

baseline = window(100, seed=1)
current  = window(100, shift=0.2, seed=2)

def stats(s):
    n=len(s); m=sum(s)/n
    std=(sum((v-m)**2 for v in s)/n)**0.5
    pos=sum(1 for v in s if v>0.5)/n
    return {"mean":round(m,3),"std":round(std,3),"pos_rate":round(pos,3)}

b,c = stats(baseline), stats(current)
print(f"{'Metric':>10}  {'Baseline':>9}  {'Current':>9}  {'Drift?':>8}")
print("-" * 45)
for key in b:
    flag = "YES" if abs(c[key]-b[key]) > 0.05 else "ok"
    print(f"{key:>10}  {b[key]:>9}  {c[key]:>9}  {flag:>8}")


    Metric   Baseline    Current    Drift?
---------------------------------------------
      mean      0.392      0.562       YES
       std      0.141      0.161        ok
  pos_rate       0.21       0.71       YES


In [233]:
# Practice: Model Deployment Basics

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## MLOps Basics

### Study checklist
- [ ] Experiment tracking
- [ ] Data versioning
- [ ] Model registry
- [ ] CI/CD for ML
- [ ] Drift detection
- [ ] Retraining strategy

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Experiment tracking

**Approach:**
- Protects reproducibility and evaluation integrity: logging hyperparameters, code version, data version, and metrics for every training run is what lets you compare experiments honestly and reproduce a past result.
- Why this is the right approach: a model's performance is a function of several inputs simultaneously, hyperparameters, code version, and data version, so reproducing or comparing any past result mathematically requires knowing every one of those inputs, not just the resulting metric - logging all of them together is what makes 'which configuration produced this number' answerable rather than a set of equations with more unknowns than recorded values.
- For this checklist item: Log every run's hyperparameters, metrics, and artifacts (model file, plots) to a central store like MLflow so experiments are comparable and reproducible.
- Code walkthrough: Check that every run's params, metrics, and artifacts are logged together, and that best is selected by comparing val_acc across all three logged runs - that comparison is only possible because every run was tracked, not just the final one.

**Learn more:**
- Website: [Google Cloud: MLOps guide](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=MLOps+Basics+Experiment+tracking+machine+learning+theory)

**Trade-offs:**
- Unlike a spreadsheet of results someone updates manually, a proper experiment tracker captures hyperparameters, code version, and metrics automatically - removing the human error and gaps that make manual tracking unreliable at scale.
- Rigorous tracking adds a small amount of friction to every experiment, but without it, knowing which run produced your best model and why becomes unanswerable once you're past a handful of ad hoc runs; the cost is trivial compared to re-running lost experiments.

**Practical software engineering use cases:**
- When to use it: Use experiment tracking from the very first tuning run, not after you've lost track of which configuration produced your best result.
- When not to use it: Don't rely on informal tracking, like notebook cell outputs or memory, once you're running more than a handful of experiments - reconstructing what produced a result becomes impossible.


In [234]:
# MLOps Basics - Experiment tracking
experiments = []

def log_run(run_id, params, metrics, artifacts=None):
    experiments.append({"run_id":run_id,"params":params,
                         "metrics":metrics,"artifacts":artifacts or []})

log_run("run_001",{"model":"LogisticRegression","C":1.0},
        {"val_acc":0.80,"f1":0.78},["model_v1.pkl"])
log_run("run_002",{"model":"RandomForest","n_estimators":100,"max_depth":5},
        {"val_acc":0.85,"f1":0.83},["model_v2.pkl","feature_importance.png"])
log_run("run_003",{"model":"GradientBoosting","n_estimators":200,"lr":0.05},
        {"val_acc":0.87,"f1":0.86},["model_v3.pkl"])

for e in experiments:
    print(f"Run: {e['run_id']}")
    print(f"  params   : {e['params']}")
    print(f"  metrics  : {e['metrics']}")
    print(f"  artifacts: {e['artifacts']}")

best = max(experiments, key=lambda e: e["metrics"]["val_acc"])
print(f"\nBest: {best['run_id']} (val_acc={best['metrics']['val_acc']})")


Run: run_001
  params   : {'model': 'LogisticRegression', 'C': 1.0}
  metrics  : {'val_acc': 0.8, 'f1': 0.78}
  artifacts: ['model_v1.pkl']
Run: run_002
  params   : {'model': 'RandomForest', 'n_estimators': 100, 'max_depth': 5}
  metrics  : {'val_acc': 0.85, 'f1': 0.83}
  artifacts: ['model_v2.pkl', 'feature_importance.png']
Run: run_003
  params   : {'model': 'GradientBoosting', 'n_estimators': 200, 'lr': 0.05}
  metrics  : {'val_acc': 0.87, 'f1': 0.86}
  artifacts: ['model_v3.pkl']

Best: run_003 (val_acc=0.87)


### Checklist item: Data versioning

**Approach:**
- Protects reproducibility and leakage prevention: versioning the exact dataset, not just the code, used to train a model is what lets you reproduce results and diagnose whether a regression came from a code change or a data change.
- Why this is the right approach: a model's fitted parameters are a deterministic, given a fixed seed, function of exactly which training data it saw - two 'the same' pipelines run against two different unversioned snapshots of a table can mathematically produce different fitted models with no code difference to explain why, precisely the ambiguity a data hash or version tag removes by making the exact training input explicitly identifiable.
- For this checklist item: Hash or version-stamp each dataset snapshot so you can trace which data trained which model â€” critical for debugging regressions and compliance.
- Code walkthrough: Check that v1 and v2 produce different hashes purely because two rows were added - the hash is a fingerprint of the exact data version, not just a row count.

**Learn more:**
- Website: [Google Cloud: MLOps guide](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=MLOps+Basics+Data+versioning+machine+learning+theory)

**Trade-offs:**
- Unlike code versioning, which is a solved, ubiquitous practice, data versioning tooling is less standardized - meaning teams often under-invest here relative to how much a silent data change can break reproducibility.
- Full data versioning, using tools like DVC or dataset snapshots, has storage and tooling overhead proportional to dataset size and change frequency; for small, rarely-changing datasets, a simpler convention such as immutable, timestamped files may be sufficient instead of a dedicated tool.

**Practical software engineering use cases:**
- When to use it: Use a dedicated data versioning tool, like DVC, once your dataset is large, frequently updated, or shared across a team.
- When not to use it: Don't skip data versioning entirely for a dataset that changes over time just because a heavyweight tool feels like overkill - at minimum, keep immutable, timestamped snapshots.


In [235]:
# MLOps Basics - Data versioning
import hashlib, json

def dataset_hash(records):
    s = json.dumps(records, sort_keys=True).encode()
    return hashlib.sha256(s).hexdigest()[:12]

v1 = [{"user":"a","score":0.9,"churned":1},
      {"user":"b","score":0.3,"churned":0},
      {"user":"c","score":0.7,"churned":1}]
v2 = v1 + [{"user":"d","score":0.4,"churned":0},
            {"user":"e","score":0.8,"churned":1}]

h1, h2 = dataset_hash(v1), dataset_hash(v2)
print(f"v1 ({len(v1)} rows) hash: {h1}")
print(f"v2 ({len(v2)} rows) hash: {h2}")
print(f"Hashes differ: {h1!=h2}  â€” confirms data changed")

log = [{"version":"v1","hash":h1,"rows":len(v1),"note":"baseline"},
       {"version":"v2","hash":h2,"rows":len(v2),"note":"added 2 rows"}]
for e in log:
    print(f"  {e['version']}: {e['rows']} rows  hash={e['hash']}  note={e['note']}")


v1 (3 rows) hash: 2abbf6161e39
v2 (5 rows) hash: 3722d66639a6
Hashes differ: True  â€” confirms data changed
  v1: 3 rows  hash=2abbf6161e39  note=baseline
  v2: 5 rows  hash=3722d66639a6  note=added 2 rows


### Checklist item: Model registry

**Approach:**
- Protects deployment reliability: a model registry is the single source of truth for which model version is staged, in production, or archived, which prevents the ambiguity of tracking that in spreadsheets, filenames, or tribal knowledge.
- Why this is the right approach: 'the current production model' is only a well-defined statement if there's a single authoritative source of truth mapping that label to one specific model version - without a registry, that mapping potentially exists differently in several places, a well-known source of state inconsistency in any distributed system, not just ML specifically.
- For this checklist item: Track which model versions are in staging vs production; record when transitions happen so you can roll back to a previous champion.
- Code walkthrough: Check that promote() changes a model's stage field in place, and that reg.show() before and after reflects v3 moving to production and v2 to archived - real state transitions, not just printed labels.

**Learn more:**
- Website: [Google Cloud: MLOps guide](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=MLOps+Basics+Model+registry+machine+learning+theory)

**Trade-offs:**
- Unlike tracking a model's status in a spreadsheet or filenames, a model registry gives every team a single, queryable source of truth for what's staged, in production, or archived - removing an entire class of which-version-is-live confusion.
- A registry adds process overhead, such as promotion steps and approvals, that can slow down iteration speed, but it's what makes rollback, auditing, and multi-team collaboration on models tractable at any real scale.

**Practical software engineering use cases:**
- When to use it: Use a model registry once more than one person or team needs to know which model version is currently deployed.
- When not to use it: Don't skip a registry for a solo, low-stakes project where you're the only one who ever needs to know which model is live - the process overhead may not be worth it yet.


In [236]:
# MLOps Basics - Model registry
class Registry:
    def __init__(self): self.models = {}
    def register(self, name, ver, metrics, stage="staging"):
        self.models[f"{name}/{ver}"] = {"name":name,"version":ver,"metrics":metrics,"stage":stage}
    def promote(self, name, ver, to):
        k=f"{name}/{ver}"
        if k in self.models:
            old=self.models[k]["stage"]; self.models[k]["stage"]=to
            print(f"  {k}: {old} â†’ {to}")
    def show(self):
        for k,m in self.models.items():
            print(f"  [{m['stage']:>10}] {k}  {m['metrics']}")

reg = Registry()
reg.register("churn","v1",{"val_f1":0.75})
reg.register("churn","v2",{"val_f1":0.82})
reg.register("churn","v3",{"val_f1":0.85})
print("Initial state:"); reg.show()
reg.promote("churn","v3","production")
reg.promote("churn","v2","archived")
print("\nAfter promotions:"); reg.show()


Initial state:
  [   staging] churn/v1  {'val_f1': 0.75}
  [   staging] churn/v2  {'val_f1': 0.82}
  [   staging] churn/v3  {'val_f1': 0.85}
  churn/v3: staging â†’ production
  churn/v2: staging â†’ archived

After promotions:
  [   staging] churn/v1  {'val_f1': 0.75}
  [  archived] churn/v2  {'val_f1': 0.82}
  [production] churn/v3  {'val_f1': 0.85}


### Checklist item: CI/CD for ML

**Approach:**
- Protects deployment reliability and evaluation integrity: automating tests, data and schema checks, and evaluation gates on every model change is what prevents a regression from reaching production the way it would for regular software.
- Why this is the right approach: an automated gate requiring new_val_metric >= champion_metric - tolerance before deployment is a precise, checkable boolean condition - that's what allows a regression to be caught deterministically, every time, versus relying on a human to remember to manually compare two numbers before every deployment, which has no such guarantee of being consistently checked.
- For this checklist item: Run automated lint, unit tests, data validation, training, model comparison, and a staging smoke-test on every PR before merging model code.
- Code walkthrough: Check that result is only DEPLOY APPROVED if every single stage's status is pass - flip any one stage to fail mentally and see that the final gate would block deployment.

**Learn more:**
- Website: [Google Cloud: MLOps guide](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=MLOps+Basics+CI%2FCD+for+ML+machine+learning+theory)

**Trade-offs:**
- Unlike CI/CD for regular software, which mostly checks for code correctness, ML CI/CD also has to check for data and performance regressions - a broader, slower kind of gate that regular software pipelines don't need.
- ML CI/CD needs more than unit tests; it needs data validation and performance-regression checks against a held-out set, which take longer to run and are harder to make deterministic than typical software tests, so pipelines need explicit budgets for how long evaluation gates are allowed to take.

**Practical software engineering use cases:**
- When to use it: Use automated data and performance-regression checks in your pipeline once a model change could plausibly ship a silent accuracy regression to production.
- When not to use it: Don't apply the exact same CI/CD pipeline you'd use for a stateless web service to an ML model without adding data-validation and evaluation gates - pure code tests won't catch a model that got worse.


In [237]:
# MLOps Basics - CI/CD for ML
stages = [
    ("Code quality",    ["flake8 lint","pytest unit tests"],                     "pass"),
    ("Data validation", ["schema check","missing < 5%","drift from last 7d"],    "pass"),
    ("Model training",  ["train on latest fold","val_f1 > 0.78"],                "pass"),
    ("Model comparison",["new val_f1 >= champion - 0.02","latency < 100ms"],     "pass"),
    ("Deploy staging",  ["shadow mode 1h","error rate < 0.1%"],                  "pass"),
]
print("ML CI/CD Pipeline:")
print("-" * 50)
for stage, checks, status in stages:
    icon = "âœ“" if status=="pass" else "âœ—"
    print(f"  [{icon}] {stage}")
    for c in checks: print(f"       - {c}")
result = all(s=="pass" for _,_,s in stages)
print(f"\nResult: {'DEPLOY APPROVED' if result else 'BLOCKED'}")


ML CI/CD Pipeline:
--------------------------------------------------
  [âœ“] Code quality
       - flake8 lint
       - pytest unit tests
  [âœ“] Data validation
       - schema check
       - missing < 5%
       - drift from last 7d
  [âœ“] Model training
       - train on latest fold
       - val_f1 > 0.78
  [âœ“] Model comparison
       - new val_f1 >= champion - 0.02
       - latency < 100ms
  [âœ“] Deploy staging
       - shadow mode 1h
       - error rate < 0.1%

Result: DEPLOY APPROVED


### Checklist item: Drift detection

**Approach:**
- Protects deployment reliability: drift detection monitors whether the production input distribution or the relationship between inputs and outputs has shifted from what the model was trained on, which is what tells you a model needs retraining before its performance visibly degrades.
- Why this is the right approach: comparing the current input distribution's summary statistics against the training distribution's is a direct, computable test of whether the model's implicit assumption, that production inputs resemble training inputs, still holds - since the model's fitted function cannot itself detect that its input distribution has changed, an external statistical comparison is mathematically necessary to catch it.
- For this checklist item: Monitor feature and prediction distributions over time; alert when KL-divergence or PSI exceeds a threshold to catch data or concept drift early.
- Code walkthrough: Check that shift, in standard-deviation units, crosses the 2.0 threshold given how far curr_age's mean has moved from train_age's - that's the concrete number the ALERT/WARNING/OK verdict is based on.

**Learn more:**
- Website: [Google Cloud: MLOps guide](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=MLOps+Basics+Drift+detection+machine+learning+theory)

**Trade-offs:**
- Unlike a bug in code, which usually causes an immediate, visible failure, model drift causes a slow, silent decline - you need dedicated monitoring specifically because nothing else will alert you to it.
- Sensitive drift detection catches degradation early but generates more false alarms that need investigation; thresholds should be tuned against how costly a missed drift event vs. an unnecessary retrain actually is for your system.

**Practical software engineering use cases:**
- When to use it: Use drift detection on any production model whose input distribution is plausibly non-stationary, like user behavior or market conditions.
- When not to use it: Don't set drift detection thresholds so sensitive that every minor fluctuation triggers an alert - investigate a couple of false alarms before tightening further, or the alerts will get ignored.


In [238]:
# MLOps Basics - Drift detection
import math

def stats(vals):
    n=len(vals); m=sum(vals)/n
    std=math.sqrt(sum((v-m)**2 for v in vals)/n)
    return m, std

train_age = [28,35,42,31,27,44,38,30,25,40]
curr_age  = [50,58,62,55,48,60,53,57,61,65]

tm,ts = stats(train_age); cm,cs = stats(curr_age)
print("Feature: 'age'")
print(f"  Training : mean={tm:.1f}  std={ts:.1f}")
print(f"  Current  : mean={cm:.1f}  std={cs:.1f}")

shift = abs(cm-tm)/ts
print(f"  Mean shift: {abs(cm-tm):.1f}  ({shift:.2f} std units)")
if shift > 2:
    print("  ALERT: significant drift â€” retrain or investigate pipeline")
elif shift > 1:
    print("  WARNING: moderate drift â€” monitor closely")
else:
    print("  OK: within normal range")


Feature: 'age'
  Training : mean=34.0  std=6.4
  Current  : mean=56.9  std=5.1
  Mean shift: 22.9  (3.59 std units)
  ALERT: significant drift â€” retrain or investigate pipeline


### Checklist item: Retraining strategy

**Approach:**
- Protects deployment reliability: deciding when and how to retrain, whether on a schedule, triggered by drift, or continuously, determines whether a model stays aligned with a changing world or slowly decays in production.
- Why this is the right approach: a model's fitted parameters reflect the data distribution AT THE TIME it was trained, a fixed snapshot - as the real-world distribution drifts away from that snapshot, the fixed function's accuracy is expected, by the same logic as drift detection, to degrade; retraining on the current distribution is the direct fix that data-drift monitoring alone can only ever detect, not correct on its own.
- For this checklist item: Define retrain triggers (schedule, performance degradation, or drift threshold) explicitly so the team doesn't retrain reactively after production failures.
- Code walkthrough: Check that 'degraded' and 'drifted' both trigger retraining but for different reasons, an f1 drop vs. a drift score, while 'stable' triggers on neither - should_retrain returning distinct reason lists per scenario.

**Learn more:**
- Website: [Google Cloud: MLOps guide](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=MLOps+Basics+Retraining+strategy+machine+learning+theory)

**Trade-offs:**
- Unlike a one-time model deployment, a retraining strategy treats the model as an ongoing process - the operational commitment that turns a single trained artifact into a maintained system.
- Frequent retraining keeps the model current but costs compute and adds risk of introducing a regression with every update; scheduled or drift-triggered retraining with a validation gate before promotion balances freshness against stability better than either fixed-interval or purely reactive retraining alone.

**Practical software engineering use cases:**
- When to use it: Use scheduled or drift-triggered retraining, gated by a validation check before promotion, once a model is live long enough that the world it was trained on can plausibly change.
- When not to use it: Don't retrain reactively only after a stakeholder notices degraded performance - by then, the model may have been quietly wrong for a while.


In [239]:
# MLOps Basics - Retraining strategy
def should_retrain(curr_f1, deploy_f1, days_since_train, drift_score):
    reasons = []
    if curr_f1 < deploy_f1 - 0.03: reasons.append(f"f1 degraded by {deploy_f1-curr_f1:.2f}")
    if days_since_train > 30: reasons.append(f"{days_since_train}d since last train")
    if drift_score > 0.20: reasons.append(f"drift={drift_score:.2f}")
    return bool(reasons), reasons

scenarios = [
    ("stable",   0.85, 0.86, 10,  0.05),
    ("degraded", 0.80, 0.86, 45,  0.18),
    ("drifted",  0.82, 0.84, 20,  0.28),
    ("stale",    0.84, 0.85, 60,  0.10),
]
print(f"{'Scenario':>10}  {'Retrain?':>8}  Reason")
print("-" * 60)
for name, curr, dep, days, drift in scenarios:
    trigger, reasons = should_retrain(curr,dep,days,drift)
    print(f"{name:>10}  {'YES' if trigger else 'no':>8}  {'; '.join(reasons) or 'none'}")


  Scenario  Retrain?  Reason
------------------------------------------------------------
    stable        no  none
  degraded       YES  f1 degraded by 0.06; 45d since last train
   drifted       YES  drift=0.28
     stale       YES  60d since last train


In [240]:
# Practice: MLOps Basics

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



# Mini Project Tracker

| Project | Topic | Dataset | Model(s) | Metric | Status | Notes |
|---|---|---|---|---|---|---|
| House Price Prediction | Regression |  | Linear Regression / Random Forest | RMSE | Not Started |  |
| Customer Churn Prediction | Classification |  | Logistic Regression / Random Forest / XGBoost | F1 / ROC-AUC | Not Started |  |
| Stock / Sales Forecasting | Time Series |  | ARIMA / ML model | MAPE / RMSE | Not Started |  |
| Text Classification | NLP |  | TF-IDF + Logistic Regression | F1 | Not Started |  |
| RAG App | GenAI | Documents | Embeddings + LLM | Grounded answer quality | Not Started |  |


# Experiment Log

Use this table to track experiments.

| Date | Problem | Dataset | Model | Features Used | Metric | Result | Next Step |
|---|---|---|---|---|---|---|---|
| YYYY-MM-DD | Churn Prediction |  | Logistic Regression | baseline | F1 |  | try Random Forest |


In [305]:
# Scratchpad

